# BetaTour v9 — canlı LKH müdahalesi, pilot deney

Bu pilot, **eski v8 örneklerinin sekizini** kullanır; sonuçları yeni bir doğrulama kümesi olarak sunmayacağız. Amaç, LKH sürecini yeniden başlatmadan TourFold tahminini aramaya eklemek ve bunun gerçekten işe yarayıp yaramadığını görmek. Altı LKH tohumu ilk turlarını ürettikten sonra aynı süreçler devam eder. Her kolun 120/300 saniyelik **duvar saati** sınırına model çıkarımı ve bekleme dahildir.

Çalıştırma: Colab'da **A100 veya L4 GPU**, High RAM ve en az **12 görünür vCPU** seçin. Pilota başladıktan sonra aynı GPU türünü ve vCPU sayısını kullanın. CPU modeli değişirse her kolun donanımı ayrı kaydedilir ve karışık örnekler analizde işaretlenir. Hücreleri sırayla çalıştırın. Her tamamlanan kol `MyDrive/BetaTour/v9_live_pilot` içine kaydolur. Oturum kesilirse yeniden çalıştırın; tamamlanan kollar atlanır, yarım kalan kol tekrarlanır. Bütün pilot bitince sonuç ZIP'i tarayıcıdan otomatik indirilir.

Kollar: kesintisiz POPMUSIC-5/6, canlı müdahalesiz kontrol, BetaTour ile mevcut adayın öne alınması, seçili düğümlere seyrek BetaTour kenarı ve aynı düğümlerde tur sıklığına dayalı kenar. Pilot politika seçimi içindir; sonraki bağımsız test ayrı bir çalışma olacaktır.


In [ ]:
EXPERIMENT_NAME = "v9_live_pilot"
SEEDS = list(range(301, 307))
ARMS = ["native_pop5", "native_pop6", "sham", "beta_promote", "beta_sparse", "frequency_same_gate"]
INSTANCE_SPECS = [
    ["uni10k", 10000, 2, "uniform", 4107, 120.0],
    ["clu10k", 10000, 2, "clustered", 4208, 120.0],
    ["uni20k", 20000, 2, "uniform", 4309, 300.0],
    ["clu20k", 20000, 2, "clustered", 4410, 300.0],
]
EXPECTED_XY_SHA256 = {
    "augment_uni10k_000": "0d5d680b465faa68d33467c4f70bb5ad1e52b25a2d444066ab922130458efcd6",
    "augment_uni10k_001": "00747175507524f481bd6e34ea55f0a5474d3dcd642a67543d871d9a64b53796",
    "augment_clu10k_000": "072fc96ec668bb9c95b3a52562f23cedb0b2e89494946d159243b638b7b3dc8e",
    "augment_clu10k_001": "965fe4315060118d313a357404f750a23ea89cc11c174cb1138a9f32b312ca22",
    "augment_uni20k_000": "d0dbad4fdd4151029d3bf59cca81c19248129c87030003d7c455e37e32b7d674",
    "augment_uni20k_001": "c35e5c135b527d91c5c9b6258320f35526c2e735a50d581c48e96150e9cb3173",
    "augment_clu20k_000": "abdd6474a4cd0e25ee394c227ef8f5c4beb5c09c8a24abddcad273d0dc39181e",
    "augment_clu20k_001": "ee3edf42dbbb94cd344d1e3bef06d5674e47330e937a8acda4fd069f4faf79ff",
}
MAX_NEW_ARMS = None  # Örn. 3: bu oturumda en fazla üç yeni kol; sonra tekrar çalıştırın.
SPARSE_FRACTION = 0.25
REPO_COMMIT = "d63f7e17942a6af5cba14ad010edca1069d477b3"
CHECKPOINT_SHA256 = "2322f66c752913f6179f8c61f6b50b0f9d21426857d3ec0155e02891fb3ebd70"
LKH_TAR_SHA256 = "4c4b9e6a77e50a1eb1eb2c8cb384ed5ef64e0dc88fdbf37bd693798469e1cf33"
LKH_FINDTOUR_SHA256 = "567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee"
assert len(SEEDS) == 6 and len(EXPECTED_XY_SHA256) == 8


In [ ]:
import base64, hashlib, importlib.util, json, math, os, platform, shutil, subprocess, sys, tarfile, time, urllib.request, uuid, zipfile
from pathlib import Path

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
for package in ("scipy", "pandas"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
import numpy as np
import scipy
from scipy.stats import binomtest
import pandas as pd
import torch
from IPython.display import display

assert torch.cuda.is_available(), "Colab GPU runtime seçin (A100 veya L4)."
torch.set_num_threads(1)
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
assert "A100" in GPU_NAME or "L4" in GPU_NAME, f"Beklenen GPU A100/L4; görünen: {GPU_NAME}"
CPU_COUNT = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text().split()[:2]
    if quota != "max":
        CPU_COUNT = min(CPU_COUNT, max(1, math.floor(int(quota) / int(period))))
assert CPU_COUNT >= 12, f"En az 12 görünür vCPU gerekli; görünen: {CPU_COUNT}"
def cpu_model_name():
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.lower().startswith("model name"):
            return line.split(":", 1)[1].strip()
    return platform.processor() or "unknown"

WORK_ROOT = Path("/content/betatour_v9_live_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REPO = WORK_ROOT / "repo"
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/kurbankotan/betatour-tsp-lkh-candidates.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() == REPO_COMMIT
assert hashlib.sha256((REPO / "checkpoints/betatour_v2.pt").read_bytes()).hexdigest() == CHECKPOINT_SHA256
sys.path.insert(0, str(REPO / "src"))
import tsplab as T
import tsp_model as M
MODEL, _ = M.load_model(str(REPO / "checkpoints/betatour_v2.pt"), DEVICE)
assert MODEL.cfg.get("use_ensemble") is True

from google.colab import drive
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/BetaTour") / EXPERIMENT_NAME
for sub in ("instances", "code", "summary", "errors", "archives"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)
HARDWARE = {"gpu": GPU_NAME, "vcpus": CPU_COUNT, "cpu_model": cpu_model_name(),
            "python": platform.python_version(), "torch": torch.__version__}
print("Hazır:", HARDWARE, "Drive:", OUTPUT_ROOT)


Mounted at /content/drive
Hazır: {'gpu': 'NVIDIA A100-SXM4-40GB', 'vcpus': 12, 'cpu_model': 'Intel(R) Xeon(R) CPU @ 2.20GHz', 'python': '3.13.16', 'torch': '2.11.0+cu130'} Drive: /content/drive/MyDrive/BetaTour/v9_live_pilot


In [ ]:
BUNDLED_FILES = {'livehook/BetaTourHook.h': 'LyogQmV0YVRvdXIgcmVzZWFyY2ggaG9vayBmb3IgTEtIIDMuMC4xNC4gIFRoaXMgZmlsZSBjb250YWlucyBvbmx5IG5ldyBjb2RlLiAqLwojaWZuZGVmIEJFVEFUT1VSX0hPT0tfSAojZGVmaW5lIEJFVEFUT1VSX0hPT0tfSAoKI2luY2x1ZGUgPGVycm5vLmg+CiNpbmNsdWRlIDxsaW1pdHMuaD4KI2luY2x1ZGUgPHN0ZGlvLmg+CiNpbmNsdWRlIDxzdGRsaWIuaD4KI2luY2x1ZGUgPHN0cmluZy5oPgojaW5jbHVkZSA8c3lzL3RpbWUuaD4KI2luY2x1ZGUgPHVuaXN0ZC5oPgoKc3RhdGljIGNvbnN0IGNoYXIgKkJUX0hvb2tEaXJlY3Rvcnkodm9pZCkKewogICAgY29uc3QgY2hhciAqUGF0aCA9IGdldGVudigiQkVUQVRPVVJfSE9PS19ESVIiKTsKICAgIHJldHVybiBSdW4gPT0gMSAmJiBQcm9ibGVtVHlwZSA9PSBUU1AgJiYgRGltZW5zaW9uID09IERpbWVuc2lvblNhdmVkICYmCiAgICAgICAgUGF0aCAmJiAqUGF0aCA/IFBhdGggOiAwOwp9CgpzdGF0aWMgdm9pZCBCVF9QYXRoKGNoYXIgKk91dCwgc2l6ZV90IFNpemUsIGNvbnN0IGNoYXIgKkRpcmVjdG9yeSwKICAgICAgICAgICAgICAgICAgICBjb25zdCBjaGFyICpOYW1lKQp7CiAgICBpZiAoc25wcmludGYoT3V0LCBTaXplLCAiJXMvJXMiLCBEaXJlY3RvcnksIE5hbWUpID49IChpbnQpIFNpemUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgcGF0aCBleGNlZWRzIFBBVEhfTUFYXG4iKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KfQoKc3RhdGljIGRvdWJsZSBCVF9XYWxsVGltZSh2b2lkKQp7CiAgICBzdHJ1Y3QgdGltZXZhbCBUdjsKICAgIGdldHRpbWVvZmRheSgmVHYsIDApOwogICAgcmV0dXJuIFR2LnR2X3NlYyArIFR2LnR2X3VzZWMgLyAxMDAwMDAwLjA7Cn0KCnN0YXRpYyBGSUxFICpCVF9PcGVuVGVtcG9yYXJ5KGNvbnN0IGNoYXIgKkRpcmVjdG9yeSwgY29uc3QgY2hhciAqTmFtZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2hhciAqRmluYWwsIGNoYXIgKlRlbXBvcmFyeSkKewogICAgRklMRSAqRmlsZTsKICAgIEJUX1BhdGgoRmluYWwsIFBBVEhfTUFYLCBEaXJlY3RvcnksIE5hbWUpOwogICAgaWYgKHNucHJpbnRmKFRlbXBvcmFyeSwgUEFUSF9NQVgsICIlcy50bXAuJWxkIiwgRmluYWwsIChsb25nKSBnZXRwaWQoKSkKICAgICAgICA+PSBQQVRIX01BWCkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayB0ZW1wb3JhcnkgcGF0aCBleGNlZWRzIFBBVEhfTUFYXG4iKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIEZpbGUgPSBmb3BlbihUZW1wb3JhcnksICJ3Iik7CiAgICBpZiAoIUZpbGUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgY2Fubm90IHdyaXRlICVzOiAlc1xuIiwgVGVtcG9yYXJ5LAogICAgICAgICAgICAgICAgc3RyZXJyb3IoZXJybm8pKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIHJldHVybiBGaWxlOwp9CgpzdGF0aWMgdm9pZCBCVF9Db21taXQoRklMRSAqRmlsZSwgY29uc3QgY2hhciAqVGVtcG9yYXJ5LCBjb25zdCBjaGFyICpGaW5hbCkKewogICAgaWYgKGZjbG9zZShGaWxlKSAhPSAwIHx8IHJlbmFtZShUZW1wb3JhcnksIEZpbmFsKSAhPSAwKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIGNhbm5vdCBjb21taXQgJXM6ICVzXG4iLCBGaW5hbCwKICAgICAgICAgICAgICAgIHN0cmVycm9yKGVycm5vKSk7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9Cn0KCi8qIEVhY2ggcm93OiBub2RlIElELCBjb3VudCwgdGhlbiAobmVpZ2hib3IgSUQsIGFscGhhKSBwYWlycy4gKi8Kc3RhdGljIHZvaWQgQlRfV3JpdGVDYW5kaWRhdGVzKGNvbnN0IGNoYXIgKkRpcmVjdG9yeSwgY29uc3QgY2hhciAqTmFtZSkKewogICAgY2hhciBGaW5hbFtQQVRIX01BWF0sIFRlbXBvcmFyeVtQQVRIX01BWF07CiAgICBGSUxFICpGaWxlID0gQlRfT3BlblRlbXBvcmFyeShEaXJlY3RvcnksIE5hbWUsIEZpbmFsLCBUZW1wb3JhcnkpOwogICAgaW50IGk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKSB7CiAgICAgICAgQ2FuZGlkYXRlICpDYW5kaWRhdGVQdHI7CiAgICAgICAgaW50IENvdW50ID0gMDsKICAgICAgICBOb2RlICpGcm9tID0gJk5vZGVTZXRbaV07CiAgICAgICAgZm9yIChDYW5kaWRhdGVQdHIgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICAgICAgICAgICBDYW5kaWRhdGVQdHIgJiYgQ2FuZGlkYXRlUHRyLT5UbzsgQ2FuZGlkYXRlUHRyKyspCiAgICAgICAgICAgIENvdW50Kys7CiAgICAgICAgZnByaW50ZihGaWxlLCAiJWQgJWQiLCBpLCBDb3VudCk7CiAgICAgICAgZm9yIChDYW5kaWRhdGVQdHIgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICAgICAgICAgICBDYW5kaWRhdGVQdHIgJiYgQ2FuZGlkYXRlUHRyLT5UbzsgQ2FuZGlkYXRlUHRyKyspCiAgICAgICAgICAgIGZwcmludGYoRmlsZSwgIiAlZCAlZCIsIENhbmRpZGF0ZVB0ci0+VG8tPklkLAogICAgICAgICAgICAgICAgICAgIENhbmRpZGF0ZVB0ci0+QWxwaGEpOwogICAgICAgIGZwdXRjKCdcbicsIEZpbGUpOwogICAgfQogICAgQlRfQ29tbWl0KEZpbGUsIFRlbXBvcmFyeSwgRmluYWwpOwp9CgpzdGF0aWMgdm9pZCBCVF9Xcml0ZUZpcnN0VG91cihjb25zdCBjaGFyICpEaXJlY3RvcnkpCnsKICAgIGNoYXIgRmluYWxbUEFUSF9NQVhdLCBUZW1wb3JhcnlbUEFUSF9NQVhdOwogICAgRklMRSAqRmlsZSA9IEJUX09wZW5UZW1wb3JhcnkoRGlyZWN0b3J5LCAiZmlyc3QudG91ciIsIEZpbmFsLCBUZW1wb3JhcnkpOwogICAgaW50IGk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKQogICAgICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBCZXR0ZXJUb3VyW2ldKTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKfQoKLyogQW4gYXRvbWljIGluY3VtYmVudCBzdXJ2aXZlcyBhbiBleHRlcm5hbCB3YWxsLWNsb2NrIGRlYWRsaW5lL3Rlcm1pbmF0aW9uLiAqLwpzdGF0aWMgdm9pZCBCVF9SZWNvcmRCZXN0KHZvaWQpCnsKICAgIGNvbnN0IGNoYXIgKkRpcmVjdG9yeSA9IEJUX0hvb2tEaXJlY3RvcnkoKTsKICAgIGNoYXIgRmluYWxbUEFUSF9NQVhdLCBUZW1wb3JhcnlbUEFUSF9NQVhdLCBFdmVudHNbUEFUSF9NQVhdOwogICAgRklMRSAqRmlsZTsKICAgIGludCBpOwogICAgaWYgKCFEaXJlY3RvcnkpCiAgICAgICAgcmV0dXJuOwogICAgRmlsZSA9IEJUX09wZW5UZW1wb3JhcnkoRGlyZWN0b3J5LCAiYmVzdC50b3VyIiwgRmluYWwsIFRlbXBvcmFyeSk7CiAgICBmcHJpbnRmKEZpbGUsICIlZFxuIiwgRGltZW5zaW9uU2F2ZWQpOwogICAgZm9yIChpID0gMTsgaSA8PSBEaW1lbnNpb25TYXZlZDsgaSsrKQogICAgICAgIGZwcmludGYoRmlsZSwgIiVkXG4iLCBCZXR0ZXJUb3VyW2ldKTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKICAgIEJUX1BhdGgoRXZlbnRzLCBzaXplb2YoRXZlbnRzKSwgRGlyZWN0b3J5LCAiYmVzdC5ldmVudHMiKTsKICAgIEZpbGUgPSBmb3BlbihFdmVudHMsICJhIik7CiAgICBpZiAoIUZpbGUpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgY2Fubm90IGFwcGVuZCAlczogJXNcbiIsIEV2ZW50cywKICAgICAgICAgICAgICAgIHN0cmVycm9yKGVycm5vKSk7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9CiAgICBmcHJpbnRmKEZpbGUsICIlLjZmICVsbGRcbiIsIEJUX1dhbGxUaW1lKCksIChsb25nIGxvbmcpIEJldHRlckNvc3QpOwogICAgaWYgKGZjbG9zZShGaWxlKSAhPSAwKSB7CiAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIGNhbm5vdCBmbHVzaCAlc1xuIiwgRXZlbnRzKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KfQoKc3RhdGljIGludCBCVF9BcHBseU9uZShpbnQgRnJvbUlkLCBpbnQgVG9JZCwgaW50IFNsb3QsIGludCBBbHBoYSwKICAgICAgICAgICAgICAgICAgICAgICBpbnQgKldhc0V4aXN0aW5nLCBpbnQgKldhc01vdmVkKQp7CiAgICBOb2RlICpGcm9tLCAqVG87CiAgICBDYW5kaWRhdGUgKlN0YXJ0LCAqQ2hvc2VuLCBUZW1wOwogICAgaW50IENvdW50ID0gMCwgSW5kZXgsIERlc3RpbmF0aW9uOwoKICAgIGlmIChGcm9tSWQgPCAxIHx8IEZyb21JZCA+IERpbWVuc2lvblNhdmVkIHx8IFRvSWQgPCAxIHx8CiAgICAgICAgVG9JZCA+IERpbWVuc2lvblNhdmVkIHx8IFRvSWQgPT0gRnJvbUlkIHx8IFNsb3QgPCAwIHx8CiAgICAgICAgQWxwaGEgPCAwIHx8IEFscGhhID09IElOVF9NQVgpCiAgICAgICAgcmV0dXJuIC0xOwogICAgRnJvbSA9ICZOb2RlU2V0W0Zyb21JZF07CiAgICBUbyA9ICZOb2RlU2V0W1RvSWRdOwogICAgU3RhcnQgPSBGcm9tLT5DYW5kaWRhdGVTZXQ7CiAgICBmb3IgKENob3NlbiA9IFN0YXJ0OyBDaG9zZW4gJiYgQ2hvc2VuLT5UbzsgQ2hvc2VuKyspCiAgICAgICAgaWYgKENob3Nlbi0+VG8gPT0gVG8pCiAgICAgICAgICAgICpXYXNFeGlzdGluZyA9IDE7CiAgICAvKiBBIHJhbmstb25seSBhY3Rpb24gbXVzdCByZXRhaW4gdGhlIG5hdGl2ZSBlZGdlJ3MgYWxwaGEgdmFsdWUuICovCiAgICBpZiAoISpXYXNFeGlzdGluZyAmJiAhQWRkQ2FuZGlkYXRlKEZyb20sIFRvLCBDKEZyb20sIFRvKSwgQWxwaGEpKQogICAgICAgIHJldHVybiAtMTsKICAgIFN0YXJ0ID0gRnJvbS0+Q2FuZGlkYXRlU2V0OwogICAgZm9yIChDaG9zZW4gPSBTdGFydDsgQ2hvc2VuICYmIENob3Nlbi0+VG87IENob3NlbisrKQogICAgICAgIENvdW50Kys7CiAgICBmb3IgKEluZGV4ID0gMDsgSW5kZXggPCBDb3VudDsgSW5kZXgrKykKICAgICAgICBpZiAoU3RhcnRbSW5kZXhdLlRvID09IFRvKQogICAgICAgICAgICBicmVhazsKICAgIGlmIChJbmRleCA9PSBDb3VudCkKICAgICAgICByZXR1cm4gLTE7CiAgICBpZiAoU2xvdCA+IDApIHsKICAgICAgICBEZXN0aW5hdGlvbiA9IFNsb3QgLSAxOwogICAgICAgIGlmIChEZXN0aW5hdGlvbiA+PSBDb3VudCkKICAgICAgICAgICAgRGVzdGluYXRpb24gPSBDb3VudCAtIDE7CiAgICAgICAgaWYgKEluZGV4ICE9IERlc3RpbmF0aW9uKSB7CiAgICAgICAgICAgIGludCBqOwogICAgICAgICAgICBUZW1wID0gU3RhcnRbSW5kZXhdOwogICAgICAgICAgICBpZiAoSW5kZXggPiBEZXN0aW5hdGlvbikKICAgICAgICAgICAgICAgIGZvciAoaiA9IEluZGV4OyBqID4gRGVzdGluYXRpb247IGotLSkKICAgICAgICAgICAgICAgICAgICBTdGFydFtqXSA9IFN0YXJ0W2ogLSAxXTsKICAgICAgICAgICAgZWxzZQogICAgICAgICAgICAgICAgZm9yIChqID0gSW5kZXg7IGogPCBEZXN0aW5hdGlvbjsgaisrKQogICAgICAgICAgICAgICAgICAgIFN0YXJ0W2pdID0gU3RhcnRbaiArIDFdOwogICAgICAgICAgICBTdGFydFtEZXN0aW5hdGlvbl0gPSBUZW1wOwogICAgICAgICAgICAqV2FzTW92ZWQgPSAxOwogICAgICAgIH0KICAgIH0KICAgIHJldHVybiAwOwp9CgpzdGF0aWMgdm9pZCBCVF9XYWl0QW5kQXBwbHkoY29uc3QgY2hhciAqRGlyZWN0b3J5KQp7CiAgICBjaGFyIEFjdGlvbltQQVRIX01BWF0sIEZpbmFsW1BBVEhfTUFYXSwgVGVtcG9yYXJ5W1BBVEhfTUFYXTsKICAgIEZJTEUgKkZpbGU7CiAgICBpbnQgTiwgaSwgRnJvbSwgVG8sIFNsb3QsIEFscGhhOwogICAgaW50IEFkZGVkID0gMCwgRXhpc3RpbmcgPSAwLCBSZWplY3RlZCA9IDAsIE1vdmVkID0gMDsKICAgIGRvdWJsZSBTdGFydCA9IEJUX1dhbGxUaW1lKCksIFRpbWVvdXQgPSAzMDAuMDsKICAgIGNvbnN0IGNoYXIgKlRpbWVvdXRUZXh0ID0gZ2V0ZW52KCJCRVRBVE9VUl9IT09LX1RJTUVPVVRfUyIpOwogICAgaWYgKFRpbWVvdXRUZXh0ICYmIGF0b2YoVGltZW91dFRleHQpID4gMCkKICAgICAgICBUaW1lb3V0ID0gYXRvZihUaW1lb3V0VGV4dCk7CiAgICBCVF9QYXRoKEFjdGlvbiwgc2l6ZW9mKEFjdGlvbiksIERpcmVjdG9yeSwgImFjdGlvbi50eHQiKTsKICAgIHdoaWxlICghKEZpbGUgPSBmb3BlbihBY3Rpb24sICJyIikpKSB7CiAgICAgICAgaWYgKEJUX1dhbGxUaW1lKCkgLSBTdGFydCA+IFRpbWVvdXQpIHsKICAgICAgICAgICAgZnByaW50ZihzdGRlcnIsICJCZXRhVG91ciBob29rIHRpbWVkIG91dCB3YWl0aW5nIGZvciAlc1xuIiwgQWN0aW9uKTsKICAgICAgICAgICAgZXhpdCg5Nyk7CiAgICAgICAgfQogICAgICAgIHVzbGVlcCgxMDAwMCk7CiAgICB9CiAgICBpZiAoZnNjYW5mKEZpbGUsICIlZCIsICZOKSAhPSAxIHx8IE4gIT0gRGltZW5zaW9uU2F2ZWQpIHsKICAgICAgICBmcHJpbnRmKHN0ZGVyciwgIkJldGFUb3VyIGhvb2sgYWN0aW9uIGRpbWVuc2lvbiBtaXNtYXRjaFxuIik7CiAgICAgICAgZXhpdCg5Nyk7CiAgICB9CiAgICBmb3IgKGkgPSAxOyBpIDw9IE47IGkrKykgewogICAgICAgIGludCBXYXNFeGlzdGluZyA9IDAsIFdhc01vdmVkID0gMDsKICAgICAgICBpZiAoZnNjYW5mKEZpbGUsICIlZCAlZCAlZCAlZCIsICZGcm9tLCAmVG8sICZTbG90LCAmQWxwaGEpICE9IDQgfHwKICAgICAgICAgICAgRnJvbSAhPSBpKSB7CiAgICAgICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayBtYWxmb3JtZWQgYWN0aW9uIHJvdyAlZFxuIiwgaSk7CiAgICAgICAgICAgIGV4aXQoOTcpOwogICAgICAgIH0KICAgICAgICBpZiAoVG8gPT0gMCkKICAgICAgICAgICAgY29udGludWU7CiAgICAgICAgaWYgKEJUX0FwcGx5T25lKEZyb20sIFRvLCBTbG90LCBBbHBoYSwgJldhc0V4aXN0aW5nLAogICAgICAgICAgICAgICAgICAgICAgICAmV2FzTW92ZWQpICE9IDApCiAgICAgICAgICAgIFJlamVjdGVkKys7CiAgICAgICAgZWxzZSB7CiAgICAgICAgICAgIEFkZGVkICs9ICFXYXNFeGlzdGluZzsKICAgICAgICAgICAgRXhpc3RpbmcgKz0gV2FzRXhpc3Rpbmc7CiAgICAgICAgICAgIE1vdmVkICs9IFdhc01vdmVkOwogICAgICAgIH0KICAgIH0KICAgIGZjbG9zZShGaWxlKTsKICAgIGlmIChSZWplY3RlZCkgewogICAgICAgIGZwcmludGYoc3RkZXJyLCAiQmV0YVRvdXIgaG9vayByZWplY3RlZCAlZCBjYW5kaWRhdGUgYWN0aW9uc1xuIiwKICAgICAgICAgICAgICAgIFJlamVjdGVkKTsKICAgICAgICBleGl0KDk3KTsKICAgIH0KICAgIEJUX1dyaXRlQ2FuZGlkYXRlcyhEaXJlY3RvcnksICJwb3N0LmNhbmRpZGF0ZXMiKTsKICAgIEZpbGUgPSBCVF9PcGVuVGVtcG9yYXJ5KERpcmVjdG9yeSwgImFwcGxpZWQuc3RhdHMiLCBGaW5hbCwgVGVtcG9yYXJ5KTsKICAgIGZwcmludGYoRmlsZSwgImFkZGVkICVkXG5leGlzdGluZyAlZFxubW92ZWQgJWRcbnJlamVjdGVkICVkXG53YWl0X3dhbGxfcyAlLjZmXG4iLAogICAgICAgICAgICBBZGRlZCwgRXhpc3RpbmcsIE1vdmVkLCBSZWplY3RlZCwgQlRfV2FsbFRpbWUoKSAtIFN0YXJ0KTsKICAgIEJUX0NvbW1pdChGaWxlLCBUZW1wb3JhcnksIEZpbmFsKTsKfQoKc3RhdGljIHZvaWQgQlRfRXhwb3J0TmF0aXZlQ2FuZGlkYXRlcyh2b2lkKQp7CiAgICBjb25zdCBjaGFyICpEaXJlY3RvcnkgPSBCVF9Ib29rRGlyZWN0b3J5KCk7CiAgICBpZiAoRGlyZWN0b3J5KQogICAgICAgIEJUX1dyaXRlQ2FuZGlkYXRlcyhEaXJlY3RvcnksICJuYXRpdmUuY2FuZGlkYXRlcyIpOwp9CgpzdGF0aWMgdm9pZCBCVF9BZnRlckZpcnN0VHJpYWwodm9pZCkKewogICAgY29uc3QgY2hhciAqRGlyZWN0b3J5ID0gQlRfSG9va0RpcmVjdG9yeSgpOwogICAgaWYgKERpcmVjdG9yeSAmJiBUcmlhbCA9PSAxKSB7CiAgICAgICAgQlRfV3JpdGVDYW5kaWRhdGVzKERpcmVjdG9yeSwgImxpdmUuY2FuZGlkYXRlcyIpOwogICAgICAgIEJUX1dyaXRlRmlyc3RUb3VyKERpcmVjdG9yeSk7IC8qIHJlYWRpbmVzcyBtYXJrZXI7IHB1Ymxpc2hlZCBsYXN0ICovCiAgICAgICAgQlRfV2FpdEFuZEFwcGx5KERpcmVjdG9yeSk7CiAgICB9Cn0KCiNlbmRpZgo=', 'livehook/build_livehook.py': 'IiIiQnVpbGQgYSBzdGF0ZS1wcmVzZXJ2aW5nIEJldGFUb3VyIGhvb2sgZnJvbSBhbiB1bnBhY2tlZCBMS0gtMy4wLjE0IHRyZWUuCgpUaGUgTEtIIHNvdXJjZSBpcyBmZXRjaGVkIHNlcGFyYXRlbHkgYnkgdGhlIGV4cGVyaW1lbnQgbm90ZWJvb2suIFRoaXMgc2NyaXB0CnNoaXBzIG9ubHkgdGhlIEJldGFUb3VyIHBhdGNoIGFuZCBjaGVja3MgdGhlIHVwc3RyZWFtIGZpbGUgYmVmb3JlIGVkaXRpbmcgaXQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBzaHV0aWwKaW1wb3J0IHN1YnByb2Nlc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgoKVVBTVFJFQU1fRklORFRPVVJfU0hBMjU2ID0gKAogICAgIjU2N2U3MmFjYzliNzM5MjQ4OGRkMTU4YmU0ZGYwMjY3NTYwNDI0MjA0YTBlODhkMmQ5YjYyZjkyM2E3YzBlZWUiCikKCgpkZWYgX3JlcGxhY2Vfb25jZShzb3VyY2U6IHN0ciwgb2xkOiBzdHIsIG5ldzogc3RyKSAtPiBzdHI6CiAgICBjb3VudCA9IHNvdXJjZS5jb3VudChvbGQpCiAgICBpZiBjb3VudCAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBvbmUgRmluZFRvdXIgYW5jaG9yLCBmb3VuZCB7Y291bnR9OiB7b2xkWzo4MF0hcn0iKQogICAgcmV0dXJuIHNvdXJjZS5yZXBsYWNlKG9sZCwgbmV3LCAxKQoKCmRlZiBwYXRjaF9maW5kdG91cihzb3VyY2U6IHN0cikgLT4gc3RyOgogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgJyNpbmNsdWRlICJMS0guaCJcbicsCiAgICAgICAgJyNpbmNsdWRlICJMS0guaCJcbiNpbmNsdWRlICJCZXRhVG91ckhvb2suaCJcbicsCiAgICApCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKAogICAgICAgIHNvdXJjZSwKICAgICAgICAiICAgIGRvdWJsZSBFbnRyeVRpbWUgPSBHZXRUaW1lKCk7XG5cbiAgICB0ID0gRmlyc3ROb2RlOyIsCiAgICAgICAgIiAgICBkb3VibGUgRW50cnlUaW1lID0gR2V0VGltZSgpO1xuXG4iCiAgICAgICAgIiAgICBCVF9FeHBvcnROYXRpdmVDYW5kaWRhdGVzKCk7XG4gICAgdCA9IEZpcnN0Tm9kZTsiLAogICAgKQogICAgc291cmNlID0gX3JlcGxhY2Vfb25jZSgKICAgICAgICBzb3VyY2UsCiAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiIsCiAgICAgICAgIiAgICAgICAgICAgIFJlY29yZEJldHRlclRvdXIoKTtcbiAgICAgICAgICAgIEJUX1JlY29yZEJlc3QoKTtcbiIsCiAgICApCiAgICBzb3VyY2UgPSBfcmVwbGFjZV9vbmNlKAogICAgICAgIHNvdXJjZSwKICAgICAgICAiICAgICAgICB9XG4gICAgfVxuICAgIGlmIChCYWNrYm9uZVRyaWFscyA+IDAgJiYgQmFja2JvbmVUcmlhbHMgPCBNYXhUcmlhbHMpIHsiLAogICAgICAgICIgICAgICAgIH1cbiAgICAgICAgQlRfQWZ0ZXJGaXJzdFRyaWFsKCk7XG4iCiAgICAgICAgIiAgICB9XG4gICAgaWYgKEJhY2tib25lVHJpYWxzID4gMCAmJiBCYWNrYm9uZVRyaWFscyA8IE1heFRyaWFscykgeyIsCiAgICApCiAgICByZXR1cm4gc291cmNlCgoKZGVmIGJ1aWxkKHNvdXJjZV9kaXI6IFBhdGgsIG91dHB1dF9kaXI6IFBhdGgsIGpvYnM6IGludCA9IDQpIC0+IFBhdGg6CiAgICBzb3VyY2VfZGlyID0gc291cmNlX2Rpci5yZXNvbHZlKCkKICAgIG91dHB1dF9kaXIgPSBvdXRwdXRfZGlyLnJlc29sdmUoKQogICAgdXBzdHJlYW0gPSBzb3VyY2VfZGlyIC8gIlNSQyIgLyAiRmluZFRvdXIuYyIKICAgIGlmIG5vdCB1cHN0cmVhbS5leGlzdHMoKSBvciBub3QgKHNvdXJjZV9kaXIgLyAiTWFrZWZpbGUiKS5leGlzdHMoKToKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkxLSC0zLjAuMTQgc291cmNlIHRyZWUgbWlzc2luZzoge3NvdXJjZV9kaXJ9IikKICAgIG9yaWdpbmFsX2J5dGVzID0gdXBzdHJlYW0ucmVhZF9ieXRlcygpCiAgICBkaWdlc3QgPSBoYXNobGliLnNoYTI1NihvcmlnaW5hbF9ieXRlcykuaGV4ZGlnZXN0KCkKICAgIGlmIGRpZ2VzdCAhPSBVUFNUUkVBTV9GSU5EVE9VUl9TSEEyNTY6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVuZXhwZWN0ZWQgTEtIIEZpbmRUb3VyLmMgU0hBMjU2OiB7ZGlnZXN0fSIpCiAgICBpZiBvdXRwdXRfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlJlZnVzaW5nIHRvIG92ZXJ3cml0ZSB7b3V0cHV0X2Rpcn0iKQogICAgc2h1dGlsLmNvcHl0cmVlKAogICAgICAgIHNvdXJjZV9kaXIsCiAgICAgICAgb3V0cHV0X2RpciwKICAgICAgICBpZ25vcmU9c2h1dGlsLmlnbm9yZV9wYXR0ZXJucygiT0JKIiwgIkxLSCIsICIqLm8iKSwKICAgICkKICAgIChvdXRwdXRfZGlyIC8gIlNSQyIgLyAiT0JKIikubWtkaXIoZXhpc3Rfb2s9VHJ1ZSkKICAgIHBhdGNoZWQgPSBwYXRjaF9maW5kdG91cihvcmlnaW5hbF9ieXRlcy5kZWNvZGUoInV0Zi04IikpCiAgICAob3V0cHV0X2RpciAvICJTUkMiIC8gIkZpbmRUb3VyLmMiKS53cml0ZV90ZXh0KHBhdGNoZWQpCiAgICBob29rID0gUGF0aChfX2ZpbGVfXykud2l0aF9uYW1lKCJCZXRhVG91ckhvb2suaCIpCiAgICBzaHV0aWwuY29weTIoaG9vaywgb3V0cHV0X2RpciAvICJTUkMiIC8gIkJldGFUb3VySG9vay5oIikKICAgIHJlc3VsdCA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgIFsibWFrZSIsICItaiIsIHN0cihqb2JzKV0sCiAgICAgICAgY3dkPW91dHB1dF9kaXIsCiAgICAgICAgc3Rkb3V0PXN1YnByb2Nlc3MuUElQRSwKICAgICAgICBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsCiAgICAgICAgdGV4dD1UcnVlLAogICAgICAgIGNoZWNrPUZhbHNlLAogICAgKQogICAgaWYgcmVzdWx0LnJldHVybmNvZGU6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTEtIIGhvb2sgYnVpbGQgZmFpbGVkOlxue3Jlc3VsdC5zdGRvdXRbLTYwMDA6XX0iKQogICAgYmluYXJ5ID0gb3V0cHV0X2RpciAvICJMS0giCiAgICBpZiBub3QgYmluYXJ5LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUGF0Y2hlZCBiaW5hcnkgbWlzc2luZzoge2JpbmFyeX0iKQogICAgbWFuaWZlc3QgPSB7CiAgICAgICAgInVwc3RyZWFtX2ZpbmR0b3VyX3NoYTI1NiI6IGRpZ2VzdCwKICAgICAgICAicGF0Y2hfaGVhZGVyX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGhvb2sucmVhZF9ieXRlcygpKS5oZXhkaWdlc3QoKSwKICAgICAgICAiYmluYXJ5X3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGJpbmFyeS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgfQogICAgKG91dHB1dF9kaXIgLyAiYmV0YXRvdXJfYnVpbGQuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAganNvbi5kdW1wcyhtYW5pZmVzdCwgaW5kZW50PTIpICsgIlxuIgogICAgKQogICAgcmV0dXJuIGJpbmFyeQoKCmRlZiBtYWluKCkgLT4gTm9uZToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWpvYnMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHByaW50KGJ1aWxkKGFyZ3Muc291cmNlX2RpciwgYXJncy5vdXRwdXRfZGlyLCBhcmdzLmpvYnMpKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK', 'livehook/coordinator.py': 'IiIiV2FsbC1jbG9jayBjb29yZGluYXRvciBmb3Igc2l4IHN0YXRlLXByZXNlcnZpbmcgQmV0YVRvdXIvTEtIIHByb2Nlc3Nlcy4KCk9ubHkgb25lIHByb2JsZW0gaW5zdGFuY2Ugc2hvdWxkIHVzZSBhIGdpdmVuIHdvcmsgZGlyZWN0b3J5LiBUaGUgYWN0aW9uCmNhbGxiYWNrIHJlY2VpdmVzIHRoZSBzaXggZmlyc3QgdG91cnMgYW5kIG5hdGl2ZS9saXZlIGNhbmRpZGF0ZSBzbmFwc2hvdHMuCkl0IHJldHVybnMgb25lIHBlci1ub2RlIGFjdGlvbiBsaXN0IHBlciBzZWVkOyBlYWNoIGFjdGlvbiBpcwpgYChuZWlnaGJvcl9pZF8xX2Jhc2VkX29yXzAsIHNsb3RfMV9iYXNlZF9vcl8wLCBhbHBoYSlgYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIFNlcXVlbmNlCgoKQGRhdGFjbGFzcwpjbGFzcyBTbmFwc2hvdDoKICAgIHNlZWQ6IGludAogICAgZmlyc3RfdG91cjogbGlzdFtpbnRdCiAgICBuYXRpdmVfY2FuZGlkYXRlczogbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dCiAgICBsaXZlX2NhbmRpZGF0ZXM6IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludF1dXQoKCmRlZiByZWFkX3RvdXIocGF0aDogUGF0aCkgLT4gbGlzdFtpbnRdOgogICAgd29yZHMgPSBbaW50KHgpIGZvciB4IGluIHBhdGgucmVhZF90ZXh0KCkuc3BsaXQoKV0KICAgIGlmIG5vdCB3b3JkcyBvciBsZW4od29yZHMpICE9IHdvcmRzWzBdICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTWFsZm9ybWVkIHRvdXI6IHtwYXRofSIpCiAgICBuID0gd29yZHNbMF0KICAgIHRvdXIgPSB3b3Jkc1sxOl0KICAgIGlmIHNvcnRlZCh0b3VyKSAhPSBsaXN0KHJhbmdlKDEsIG4gKyAxKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRvdXIgaXMgbm90IGEgcGVybXV0YXRpb246IHtwYXRofSIpCiAgICByZXR1cm4gdG91cgoKCmRlZiByZWFkX2NhbmRpZGF0ZXMocGF0aDogUGF0aCkgLT4gbGlzdFtsaXN0W3R1cGxlW2ludCwgaW50XV1dOgogICAgbGluZXMgPSBwYXRoLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKQogICAgaWYgbm90IGxpbmVzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFbXB0eSBjYW5kaWRhdGUgc25hcHNob3Q6IHtwYXRofSIpCiAgICBuID0gaW50KGxpbmVzWzBdKQogICAgaWYgbGVuKGxpbmVzKSAhPSBuICsgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ2FuZGlkYXRlIHNuYXBzaG90IGhhcyB7bGVuKGxpbmVzKS0xfSByb3dzLCBuZWVkIHtufSIpCiAgICByb3dzOiBsaXN0W2xpc3RbdHVwbGVbaW50LCBpbnRdXV0gPSBbXQogICAgZm9yIGV4cGVjdGVkX2lkLCBsaW5lIGluIGVudW1lcmF0ZShsaW5lc1sxOl0sIDEpOgogICAgICAgIG51bXMgPSBbaW50KHgpIGZvciB4IGluIGxpbmUuc3BsaXQoKV0KICAgICAgICBpZiBsZW4obnVtcykgPCAyIG9yIG51bXNbMF0gIT0gZXhwZWN0ZWRfaWQgb3IgbGVuKG51bXMpICE9IDIgKyAyICogbnVtc1sxXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIk1hbGZvcm1lZCBjYW5kaWRhdGUgcm93IHtleHBlY3RlZF9pZH06IHtwYXRofSIpCiAgICAgICAgcm93ID0gWyhudW1zW2ldLCBudW1zW2kgKyAxXSkgZm9yIGkgaW4gcmFuZ2UoMiwgbGVuKG51bXMpLCAyKV0KICAgICAgICBpZiBhbnkodiA8IDEgb3IgdiA+IG4gb3IgdiA9PSBleHBlY3RlZF9pZCBmb3IgdiwgXyBpbiByb3cpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBuZWlnaGJvciBpbiByb3cge2V4cGVjdGVkX2lkfToge3BhdGh9IikKICAgICAgICByb3dzLmFwcGVuZChyb3cpCiAgICByZXR1cm4gcm93cwoKCmRlZiBub29wX2FjdGlvbnMobjogaW50KSAtPiBsaXN0W3R1cGxlW2ludCwgaW50LCBpbnRdXToKICAgIHJldHVybiBbKDAsIDAsIDUwMCldICogbgoKCmRlZiB3cml0ZV9hY3Rpb25zKHBhdGg6IFBhdGgsIGFjdGlvbnM6IFNlcXVlbmNlW3R1cGxlW2ludCwgaW50LCBpbnRdXSkgLT4gTm9uZToKICAgIG4gPSBsZW4oYWN0aW9ucykKICAgIGxpbmVzID0gW3N0cihuKV0KICAgIGZvciBzb3VyY2VfaWQsIChuZWlnaGJvciwgc2xvdCwgYWxwaGEpIGluIGVudW1lcmF0ZShhY3Rpb25zLCAxKToKICAgICAgICBpZiBub3QgKDAgPD0gbmVpZ2hib3IgPD0gbiBhbmQgMCA8PSBzbG90IDw9IG4gKyAyIGFuZCAwIDw9IGFscGhhIDwgMioqMzEgLSAxKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgYWN0aW9uIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgaWYgbmVpZ2hib3IgPT0gc291cmNlX2lkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiU2VsZi1lZGdlIGF0IHNvdXJjZSB7c291cmNlX2lkfSIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYie3NvdXJjZV9pZH0ge25laWdoYm9yfSB7c2xvdH0ge2FscGhhfSIpCiAgICB0ZW1wb3JhcnkgPSBwYXRoLndpdGhfc3VmZml4KHBhdGguc3VmZml4ICsgIi50bXAiKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoIlxuIi5qb2luKGxpbmVzKSArICJcbiIpCiAgICBvcy5yZXBsYWNlKHRlbXBvcmFyeSwgcGF0aCkKCgpkZWYgX3BhcmFtZXRlcl9maWxlKHBhdGg6IFBhdGgsIHByb2JsZW06IFBhdGgsIHNlZWQ6IGludCwgd2FsbF9idWRnZXRfczogZmxvYXQsCiAgICAgICAgICAgICAgICAgICAgbWF4X3RyaWFsczogaW50LCBleHRyYTogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSkgLT4gTm9uZToKICAgIHBhcmFtczogZGljdFtzdHIsIHN0ciB8IGludCB8IGZsb2F0XSA9IHsKICAgICAgICAiUFJPQkxFTV9GSUxFIjogc3RyKHByb2JsZW0ucmVzb2x2ZSgpKSwKICAgICAgICAiQ0FORElEQVRFX1NFVF9UWVBFIjogIlBPUE1VU0lDIiwKICAgICAgICAiTUFYX0NBTkRJREFURVMiOiA1LAogICAgICAgICJTVUJHUkFESUVOVCI6ICJOTyIsCiAgICAgICAgIlNFRUQiOiBzZWVkLAogICAgICAgICJSVU5TIjogMSwKICAgICAgICAiTUFYX1RSSUFMUyI6IG1heF90cmlhbHMsCiAgICAgICAgIyBMS0gncyBUSU1FX0xJTUlUIG1lYXN1cmVzIHByb2Nlc3MgQ1BVIHRpbWUuIFRoZSBjb29yZGluYXRvciBiZWxvdwogICAgICAgICMgZW5mb3JjZXMgdGhlIGFjdHVhbCBlbmQtdG8tZW5kIHdhbGwgYnVkZ2V0LgogICAgICAgICJUSU1FX0xJTUlUIjogbWF4KDEwLjAsIDEwLjAgKiB3YWxsX2J1ZGdldF9zKSwKICAgICAgICAiVFJBQ0VfTEVWRUwiOiAwLAogICAgICAgICJTVE9QX0FUX09QVElNVU0iOiAiTk8iLAogICAgfQogICAgcGFyYW1zLnVwZGF0ZShleHRyYSkKICAgIHBhdGgud3JpdGVfdGV4dCgiXG4iLmpvaW4oZiJ7a2V5fSA9IHt2YWx1ZX0iIGZvciBrZXksIHZhbHVlIGluIHBhcmFtcy5pdGVtcygpKSArICJcbiIpCgoKZGVmIHJ1bl9zaXgoCiAgICBsa2hfYmluYXJ5OiBQYXRoLAogICAgcHJvYmxlbTogUGF0aCwKICAgIHdvcmtfZGlyOiBQYXRoLAogICAgc2VlZHM6IFNlcXVlbmNlW2ludF0sCiAgICB3YWxsX2J1ZGdldF9zOiBmbG9hdCwKICAgIGFjdGlvbl9mbjogQ2FsbGFibGVbW2xpc3RbU25hcHNob3RdXSwgU2VxdWVuY2VbU2VxdWVuY2VbdHVwbGVbaW50LCBpbnQsIGludF1dXV0sCiAgICAqLAogICAgbWF4X3RyaWFsczogaW50ID0gMTAwXzAwMF8wMDAsCiAgICBleHRyYV9wYXJhbXM6IGRpY3Rbc3RyLCBzdHIgfCBpbnQgfCBmbG9hdF0gfCBOb25lID0gTm9uZSwKKSAtPiBkaWN0OgogICAgIiIiUnVuIG9uZSBpbnN0YW5jZSB3aXRoIHNpeCBzZWVkczsgY2hhcmdlIHN0YXJ0dXAsIGJhcnJpZXIgYW5kIHBvbGljeSB0aW1lLgoKICAgIGB3b3JrX2RpcmAgbXVzdCBiZSBuZXcgdG8gYXZvaWQgc3RhbGUgYWN0aW9uIGZpbGVzLiBSdW5zIG1heSBlbmQgYmVmb3JlIHRoZQogICAgZGVhZGxpbmUgd2hlbiBgbWF4X3RyaWFsc2AgaXMgc21hbGwuIFJlc3VsdHMgaW5jbHVkZSBhbGwgaG9vayBmaWxlcyBzbyB0aGUKICAgIGNhbGxlciBjYW4gYXVkaXQgdGhlIGFjdHVhbCBjYW5kaWRhdGUgYXJyYXlzIGFuZCBmaXJzdCB0b3Vycy4KICAgICIiIgogICAgaWYgbGVuKHNlZWRzKSAhPSA2IG9yIGxlbihzZXQoc2VlZHMpKSAhPSA2OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkV4YWN0bHkgc2l4IGRpc3RpbmN0IHNlZWRzIGFyZSByZXF1aXJlZCIpCiAgICBpZiB3YWxsX2J1ZGdldF9zIDw9IDAgb3IgbWF4X3RyaWFscyA8IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUG9zaXRpdmUgYnVkZ2V0IGFuZCB0cmlhbHMgYXJlIHJlcXVpcmVkIikKICAgIGlmIHdvcmtfZGlyLmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVFeGlzdHNFcnJvcihmIlVzZSBhIGZyZXNoIHJ1biBkaXJlY3Rvcnk6IHt3b3JrX2Rpcn0iKQogICAgd29ya19kaXIubWtkaXIocGFyZW50cz1UcnVlKQogICAgcHJvY2Vzc2VzOiBsaXN0W3N1YnByb2Nlc3MuUG9wZW5dID0gW10KICAgIG9wZW5fbG9nczogbGlzdCA9IFtdCiAgICBzdGFydF9tb25vID0gdGltZS5tb25vdG9uaWMoKQogICAgc3RhcnRfZXBvY2ggPSB0aW1lLnRpbWUoKQogICAgZGVhZGxpbmUgPSBzdGFydF9tb25vICsgd2FsbF9idWRnZXRfcwogICAgdHJ5OgogICAgICAgIGZvciBzZWVkIGluIHNlZWRzOgogICAgICAgICAgICBzZWVkX2RpciA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgc2VlZF9kaXIubWtkaXIoKQogICAgICAgICAgICBwYXJhbXMgPSBzZWVkX2RpciAvICJydW4ucGFyIgogICAgICAgICAgICBfcGFyYW1ldGVyX2ZpbGUocGFyYW1zLCBwcm9ibGVtLCBzZWVkLCB3YWxsX2J1ZGdldF9zLCBtYXhfdHJpYWxzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZXh0cmFfcGFyYW1zIG9yIHt9KQogICAgICAgICAgICBzdGRvdXQgPSAoc2VlZF9kaXIgLyAic3Rkb3V0LnR4dCIpLm9wZW4oInciKQogICAgICAgICAgICBzdGRlcnIgPSAoc2VlZF9kaXIgLyAic3RkZXJyLnR4dCIpLm9wZW4oInciKQogICAgICAgICAgICBvcGVuX2xvZ3MuZXh0ZW5kKChzdGRvdXQsIHN0ZGVycikpCiAgICAgICAgICAgIGVudiA9IGRpY3Qob3MuZW52aXJvbikKICAgICAgICAgICAgZW52WyJCRVRBVE9VUl9IT09LX0RJUiJdID0gc3RyKHNlZWRfZGlyLnJlc29sdmUoKSkKICAgICAgICAgICAgZW52WyJCRVRBVE9VUl9IT09LX1RJTUVPVVRfUyJdID0gc3RyKG1heCg1LjAsIHdhbGxfYnVkZ2V0X3MgKyA1LjApKQogICAgICAgICAgICBwcm9jZXNzZXMuYXBwZW5kKHN1YnByb2Nlc3MuUG9wZW4oCiAgICAgICAgICAgICAgICBbc3RyKGxraF9iaW5hcnkucmVzb2x2ZSgpKSwgc3RyKHBhcmFtcy5yZXNvbHZlKCkpXSwKICAgICAgICAgICAgICAgIGN3ZD1zZWVkX2Rpciwgc3Rkb3V0PXN0ZG91dCwgc3RkZXJyPXN0ZGVyciwgZW52PWVudiwKICAgICAgICAgICAgKSkKICAgICAgICBmaXJzdF9wYXRocyA9IFt3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iIC8gImZpcnN0LnRvdXIiIGZvciBzZWVkIGluIHNlZWRzXQogICAgICAgIHdoaWxlIG5vdCBhbGwocGF0aC5leGlzdHMoKSBmb3IgcGF0aCBpbiBmaXJzdF9wYXRocyk6CiAgICAgICAgICAgIGZhaWx1cmVzID0gW3AucmV0dXJuY29kZSBmb3IgcCBpbiBwcm9jZXNzZXMgaWYgcC5wb2xsKCkgbm90IGluIChOb25lLCAwKV0KICAgICAgICAgICAgaWYgZmFpbHVyZXM6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJMS0ggZXhpdGVkIGJlZm9yZSBmaXJzdC10b3VyIGJhcnJpZXI6IHtmYWlsdXJlc30iKQogICAgICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpID49IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgdGltZS5zbGVlcCgwLjAyKQogICAgICAgIHNuYXBzaG90czogbGlzdFtTbmFwc2hvdF0gPSBbXQogICAgICAgIGlmIGFsbChwYXRoLmV4aXN0cygpIGZvciBwYXRoIGluIGZpcnN0X3BhdGhzKSBhbmQgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgICAgIGJhc2UgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgICAgICAgICBzbmFwc2hvdHMuYXBwZW5kKFNuYXBzaG90KAogICAgICAgICAgICAgICAgICAgIHNlZWQ9c2VlZCwKICAgICAgICAgICAgICAgICAgICBmaXJzdF90b3VyPXJlYWRfdG91cihiYXNlIC8gImZpcnN0LnRvdXIiKSwKICAgICAgICAgICAgICAgICAgICBuYXRpdmVfY2FuZGlkYXRlcz1yZWFkX2NhbmRpZGF0ZXMoYmFzZSAvICJuYXRpdmUuY2FuZGlkYXRlcyIpLAogICAgICAgICAgICAgICAgICAgIGxpdmVfY2FuZGlkYXRlcz1yZWFkX2NhbmRpZGF0ZXMoYmFzZSAvICJsaXZlLmNhbmRpZGF0ZXMiKSwKICAgICAgICAgICAgICAgICkpCiAgICAgICAgICAgIGFjdGlvbnMgPSBhY3Rpb25fZm4oc25hcHNob3RzKQogICAgICAgICAgICBpZiBsZW4oYWN0aW9ucykgIT0gbGVuKHNlZWRzKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlBvbGljeSBtdXN0IHJldHVybiBvbmUgYWN0aW9uIG1hdHJpeCBwZXIgc2VlZCIpCiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPCBkZWFkbGluZToKICAgICAgICAgICAgICAgIGZvciBzZWVkLCBzbmFwc2hvdCwgYWN0aW9uIGluIHppcChzZWVkcywgc25hcHNob3RzLCBhY3Rpb25zKToKICAgICAgICAgICAgICAgICAgICBpZiBsZW4oYWN0aW9uKSAhPSBsZW4oc25hcHNob3QuZmlyc3RfdG91cik6CiAgICAgICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJQb2xpY3kgZGltZW5zaW9uIG1pc21hdGNoIGZvciBzZWVkIHtzZWVkfSIpCiAgICAgICAgICAgICAgICAgICAgd3JpdGVfYWN0aW9ucyh3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iIC8gImFjdGlvbi50eHQiLCBhY3Rpb24pCiAgICAgICAgd2hpbGUgYW55KHAucG9sbCgpIGlzIE5vbmUgZm9yIHAgaW4gcHJvY2Vzc2VzKSBhbmQgdGltZS5tb25vdG9uaWMoKSA8IGRlYWRsaW5lOgogICAgICAgICAgICB0aW1lLnNsZWVwKDAuMDIpCiAgICBmaW5hbGx5OgogICAgICAgIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlczoKICAgICAgICAgICAgaWYgcHJvY2Vzcy5wb2xsKCkgaXMgTm9uZToKICAgICAgICAgICAgICAgIHByb2Nlc3MudGVybWluYXRlKCkKICAgICAgICBmb3IgcHJvY2VzcyBpbiBwcm9jZXNzZXM6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHByb2Nlc3Mud2FpdCh0aW1lb3V0PTIpCiAgICAgICAgICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgICAgICAgICAgICAgcHJvY2Vzcy5raWxsKCkKICAgICAgICAgICAgICAgIHByb2Nlc3Mud2FpdCgpCiAgICAgICAgZm9yIHN0cmVhbSBpbiBvcGVuX2xvZ3M6CiAgICAgICAgICAgIHN0cmVhbS5jbG9zZSgpCiAgICByZXN1bHQgPSB7CiAgICAgICAgInNlZWRzIjogbGlzdChzZWVkcyksCiAgICAgICAgIndhbGxfYnVkZ2V0X3MiOiB3YWxsX2J1ZGdldF9zLAogICAgICAgICJ3YWxsX2VsYXBzZWRfcyI6IHRpbWUubW9ub3RvbmljKCkgLSBzdGFydF9tb25vLAogICAgICAgICJzdGFydF9lcG9jaF9zIjogc3RhcnRfZXBvY2gsCiAgICAgICAgImRlYWRsaW5lX2Vwb2NoX3MiOiBzdGFydF9lcG9jaCArIHdhbGxfYnVkZ2V0X3MsCiAgICAgICAgImJhcnJpZXJfY29tcGxldGVkIjogbGVuKHNuYXBzaG90cykgPT0gNiwKICAgICAgICAicnVucyI6IFtdLAogICAgfQogICAgZm9yIHNlZWQsIHByb2Nlc3MgaW4gemlwKHNlZWRzLCBwcm9jZXNzZXMpOgogICAgICAgIGJhc2UgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iCiAgICAgICAgYmVzdCA9IGJhc2UgLyAiYmVzdC50b3VyIgogICAgICAgIHJlc3VsdFsicnVucyJdLmFwcGVuZCh7CiAgICAgICAgICAgICJzZWVkIjogc2VlZCwKICAgICAgICAgICAgInJldHVybmNvZGUiOiBwcm9jZXNzLnJldHVybmNvZGUsCiAgICAgICAgICAgICJmaXJzdF90b3VyIjogc3RyKGJhc2UgLyAiZmlyc3QudG91ciIpIGlmIChiYXNlIC8gImZpcnN0LnRvdXIiKS5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJiZXN0X3RvdXIiOiBzdHIoYmVzdCkgaWYgYmVzdC5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJiZXN0X2V2ZW50cyI6IHN0cihiYXNlIC8gImJlc3QuZXZlbnRzIikgaWYgKGJhc2UgLyAiYmVzdC5ldmVudHMiKS5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJuYXRpdmVfY2FuZGlkYXRlcyI6IHN0cihiYXNlIC8gIm5hdGl2ZS5jYW5kaWRhdGVzIikgaWYgKGJhc2UgLyAibmF0aXZlLmNhbmRpZGF0ZXMiKS5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJsaXZlX2NhbmRpZGF0ZXMiOiBzdHIoYmFzZSAvICJsaXZlLmNhbmRpZGF0ZXMiKSBpZiAoYmFzZSAvICJsaXZlLmNhbmRpZGF0ZXMiKS5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJwb3N0X2NhbmRpZGF0ZXMiOiBzdHIoYmFzZSAvICJwb3N0LmNhbmRpZGF0ZXMiKSBpZiAoYmFzZSAvICJwb3N0LmNhbmRpZGF0ZXMiKS5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgICAgICJhcHBsaWVkX3N0YXRzIjogc3RyKGJhc2UgLyAiYXBwbGllZC5zdGF0cyIpIGlmIChiYXNlIC8gImFwcGxpZWQuc3RhdHMiKS5leGlzdHMoKSBlbHNlIE5vbmUsCiAgICAgICAgfSkKICAgICh3b3JrX2RpciAvICJydW5fc3VtbWFyeS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlc3VsdCwgaW5kZW50PTIpICsgIlxuIikKICAgIHJldHVybiByZXN1bHQKCg==', 'policy/adaptive_policy.py': 'IiIiQ29uc2VydmF0aXZlIFBPUE1VU0lDIGNhbmRpZGF0ZSBhY3Rpb25zIGZyb20gYW4gZXhpc3RpbmcgVG91ckZvbGQgcHJlZGljdGlvbi4KClRoZSBmdW5jdGlvbnMgaW4gdGhpcyBmaWxlIGRvIG5vdCBjbGFpbSB0byBlc3RpbWF0ZSBzZWFyY2ggdXRpbGl0eS4gVGhleSB0dXJuClRvdXJGb2xkJ3MgcmVmZXJlbmNlLWVkZ2Ugc2NvcmVzIGludG8gKnRlc3RhYmxlKiBpbnRlcnZlbnRpb25zIGZvciBhIGxpdmUgTEtICmV4cGVyaW1lbnQuIE5vIGFjdGlvbiBpcyB0YWtlbiB3aGVuIGFuIGVkZ2UgY2Fubm90IGJlIHNjb3JlZCByZWxpYWJseS4KCklucHV0cyB1c2UgemVyby1iYXNlZCBub2RlIElEcy4gYGBwb3BgYCBoYXMgc2hhcGUgKHNlZWRzLCBub2Rlcywga19wb3ApLCB3aXRoCmBgLTFgYCBmb3IgYW4gdW5maWxsZWQgbmF0aXZlIFBPUE1VU0lDIHNsb3QuIGBgbmVpZ2hib3Vyc2BgL2BgcHJvYmFiaWxpdGllc2BgCmhhdmUgc2hhcGUgKG5vZGVzLCBrX2dyYXBoKS4gVGhlIHNhbWUgbW9kZWwgcHJlZGljdGlvbiBtYXkgYmUgYXBwbGllZCB0byBlYWNoCnNlZWQncyBkaWZmZXJlbnQgUE9QTVVTSUMgY2FuZGlkYXRlIHNldC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBudW1weSBhcyBucAoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFNwYXJzZUFkZENvbmZpZzoKICAgICIiIkEgcHJlY29tbWl0dGVkIGRlbnNpdHkgY2FwIGFuZCBldmlkZW5jZSBydWxlIGZvciBvbmUgZXh0cmEgZWRnZSBwZXIgbm9kZS4KCiAgICBgYG1heF9mcmFjdGlvbmBgIGxpbWl0cyBpbnRlcnZlbnRpb24gdG8gdGhlIGJlc3Qtc2NvcmVkIGZyYWN0aW9uIG9mIG5vZGVzCiAgICBpbiAqZWFjaCBzZWVkKi4gYGBtaW5fc2NvcmVkX3BvcGBgIHJlcXVpcmVzIGEgbWluaW11bSBudW1iZXIgb2YgdGhlIFBPUAogICAgZWRnZXMgdG8gYmUgcHJlc2VudCBpbiBUb3VyRm9sZCdzIHNjb3JpbmcgZ3JhcGguIGBgbWluX21hcmdpbmBgIGlzIHRoZQogICAgcHJvYmFiaWxpdHkgZGlmZmVyZW5jZSBiZXR3ZWVuIHRoZSBwcm9wb3NlZCBlZGdlIGFuZCB0aGUgd2Vha2VzdCBzY29yZWQKICAgIFBPUCBlZGdlOyB0aGVzZSBwcm9iYWJpbGl0aWVzIGFyZSBub3QgYXNzdW1lZCBjYWxpYnJhdGVkLgogICAgIiIiCgogICAgbWF4X2ZyYWN0aW9uOiBmbG9hdCA9IDAuMjUKICAgIG1pbl9zY29yZWRfcG9wOiBpbnQgPSAyCiAgICBtaW5fbWFyZ2luOiBmbG9hdCA9IDAuMAoKICAgIGRlZiBfX3Bvc3RfaW5pdF9fKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgbm90IDAuMCA8PSBzZWxmLm1heF9mcmFjdGlvbiA8PSAxLjA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1heF9mcmFjdGlvbiBtdXN0IGJlIGluIFswLCAxXSIpCiAgICAgICAgaWYgc2VsZi5taW5fc2NvcmVkX3BvcCA8IDE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIm1pbl9zY29yZWRfcG9wIG11c3QgYmUgcG9zaXRpdmUiKQogICAgICAgIGlmIG5vdCBucC5pc2Zpbml0ZShzZWxmLm1pbl9tYXJnaW4pOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtaW5fbWFyZ2luIG11c3QgYmUgZmluaXRlIikKCgpAZGF0YWNsYXNzCmNsYXNzIFNwYXJzZUFkZFJlc3VsdDoKICAgICIiIk9uZSBwZXItbm9kZSBvcHRpb25hbCBhY3Rpb24gZm9yIGV2ZXJ5IGluZGVwZW5kZW50IExLSCBwcm9jZXNzLgoKICAgIGBgZGVzdGluYXRpb24gPT0gLTFgYCBtZWFucyBrZWVwIHRoZSBuYXRpdmUgUE9QTVVTSUMgY2FuZGlkYXRlIHJvdy4KICAgIGBgcHJvcG9zZWRfZGVzdGluYXRpb25gYCByZWNvcmRzIGEgcG9zc2libGUgZ3JhcGgtbm92ZWwgZWRnZSBldmVuIHdoZW4gdGhlCiAgICBnYXRlIGRlY2xpbmVzIGl0LiBgYHNjb3JlYGAgaXMgYSBwcm94eSwgbm90IGFuIG9ic2VydmVkIExLSCBnYWluLgogICAgIiIiCgogICAgZGVzdGluYXRpb246IG5wLm5kYXJyYXkKICAgIHByb3Bvc2VkX2Rlc3RpbmF0aW9uOiBucC5uZGFycmF5CiAgICBwcm9wb3NlZF9wcm9iYWJpbGl0eTogbnAubmRhcnJheQogICAgc2NvcmU6IG5wLm5kYXJyYXkKICAgIHNjb3JlZF9wb3BfY291bnQ6IG5wLm5kYXJyYXkKICAgIGVsaWdpYmxlOiBucC5uZGFycmF5CgoKZGVmIF92YWxpZGF0ZShwb3A6IG5wLm5kYXJyYXksIG5laWdoYm91cnM6IG5wLm5kYXJyYXksIHByb2JhYmlsaXRpZXM6IG5wLm5kYXJyYXkpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgcG9wID0gbnAuYXNhcnJheShwb3ApCiAgICBuZWlnaGJvdXJzID0gbnAuYXNhcnJheShuZWlnaGJvdXJzKQogICAgcHJvYmFiaWxpdGllcyA9IG5wLmFzYXJyYXkocHJvYmFiaWxpdGllcywgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIGlmIHBvcC5uZGltICE9IDMgb3IgbmVpZ2hib3Vycy5uZGltICE9IDIgb3IgcHJvYmFiaWxpdGllcy5zaGFwZSAhPSBuZWlnaGJvdXJzLnNoYXBlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImV4cGVjdGVkIHBvcD0oc2VlZHMsbm9kZXMsayksIG5laWdoYm91cnM9cHJvYmFiaWxpdGllcz0obm9kZXMsa19ncmFwaCkiKQogICAgc2VlZHMsIG4sIGtfcG9wID0gcG9wLnNoYXBlCiAgICBpZiBzZWVkcyA8IDEgb3IgbiA8IDMgb3Iga19wb3AgPCAxIG9yIG5laWdoYm91cnMuc2hhcGVbMF0gIT0gbiBvciBuZWlnaGJvdXJzLnNoYXBlWzFdIDwgMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJpbnZhbGlkIGFycmF5IGRpbWVuc2lvbnMiKQogICAgaWYga19wb3AgPj0gbiBvciBuZWlnaGJvdXJzLnNoYXBlWzFdID49IG46CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiY2FuZGlkYXRlIHdpZHRoIG11c3QgYmUgYmVsb3cgbm9kZSBjb3VudCIpCiAgICBpZiBub3QgbnAuaXNzdWJkdHlwZShwb3AuZHR5cGUsIG5wLmludGVnZXIpIG9yIG5vdCBucC5pc3N1YmR0eXBlKG5laWdoYm91cnMuZHR5cGUsIG5wLmludGVnZXIpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNhbmRpZGF0ZSBJRHMgbXVzdCBiZSBpbnRlZ2VycyIpCiAgICBpZiBucC5hbnkoKHBvcCA8IC0xKSB8IChwb3AgPj0gbikpIG9yIG5wLmFueSgobmVpZ2hib3VycyA8IDApIHwgKG5laWdoYm91cnMgPj0gbikpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNhbmRpZGF0ZSBJRHMgb3V0c2lkZSBpbnN0YW5jZSIpCiAgICBvcmlnaW4gPSBucC5hcmFuZ2UobikKICAgIGlmIG5wLmFueShwb3AgPT0gb3JpZ2luW05vbmUsIDosIE5vbmVdKSBvciBucC5hbnkobmVpZ2hib3VycyA9PSBvcmlnaW5bOiwgTm9uZV0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNlbGYtZWRnZSBpbiBjYW5kaWRhdGVzIikKICAgICMgTXVsdGlwbGUgLTEgc2xvdHMgYXJlIGxlZ2l0aW1hdGUuIFRyZWF0IHRoZW0gYXMgYSBzZW50aW5lbCBhYm92ZSBhbGwKICAgICMgbm9kZSBJRHMgb25seSBmb3IgZHVwbGljYXRlIGRldGVjdGlvbjsgbmV2ZXIgdHVybiB0aGVtIGludG8gY2FuZGlkYXRlcy4KICAgIHNvcnRlZF9wb3AgPSBucC5zb3J0KG5wLndoZXJlKHBvcCA+PSAwLCBwb3AsIG4pLCBheGlzPTIpCiAgICBpZiBucC5hbnkoKHNvcnRlZF9wb3BbOiwgOiwgMTpdID09IHNvcnRlZF9wb3BbOiwgOiwgOi0xXSkKICAgICAgICAgICAgICAmIChzb3J0ZWRfcG9wWzosIDosIDE6XSAhPSBuKSk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZHVwbGljYXRlIFBPUCBjYW5kaWRhdGUgaW4gYSByb3ciKQogICAgaWYgbnAuYW55KG5wLnNvcnQobmVpZ2hib3VycywgYXhpcz0xKVs6LCAxOl0gPT0gbnAuc29ydChuZWlnaGJvdXJzLCBheGlzPTEpWzosIDotMV0pOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImR1cGxpY2F0ZSBtb2RlbCBncmFwaCBuZWlnaGJvdXIgaW4gYSByb3ciKQogICAgaWYgbm90IG5wLmFsbChucC5pc2Zpbml0ZShwcm9iYWJpbGl0aWVzKSkgb3IgbnAuYW55KChwcm9iYWJpbGl0aWVzIDwgMCkgfCAocHJvYmFiaWxpdGllcyA+IDEpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJwcm9iYWJpbGl0aWVzIG11c3QgYmUgZmluaXRlIGFuZCBpbiBbMCwgMV0iKQogICAgcmV0dXJuIHBvcC5hc3R5cGUobnAuaW50MzIsIGNvcHk9RmFsc2UpLCBuZWlnaGJvdXJzLmFzdHlwZShucC5pbnQzMiwgY29weT1GYWxzZSksIHByb2JhYmlsaXRpZXMKCgpkZWYgX3BvcF9zY29yZXMocG9wX3JvdzogbnAubmRhcnJheSwgbmVpZ2hib3VyczogbnAubmRhcnJheSwgcHJvYmFiaWxpdGllczogbnAubmRhcnJheSkgLT4gdHVwbGVbbnAubmRhcnJheSwgbnAubmRhcnJheV06CiAgICBtYXRjaGVzID0gcG9wX3Jvd1s6LCA6LCBOb25lXSA9PSBuZWlnaGJvdXJzWzosIE5vbmUsIDpdCiAgICBwcmVzZW50ID0gbWF0Y2hlcy5hbnkoYXhpcz0yKQogICAgc2NvcmVzID0gbnAud2hlcmUocHJlc2VudCwgKG1hdGNoZXMgKiBwcm9iYWJpbGl0aWVzWzosIE5vbmUsIDpdKS5zdW0oYXhpcz0yKSwgbnAubmFuKQogICAgcmV0dXJuIHNjb3JlcywgcHJlc2VudAoKCmRlZiByZXJhbmtfc2NvcmVkX3BvcChwb3A6IG5wLm5kYXJyYXksIG5laWdoYm91cnM6IG5wLm5kYXJyYXksIHByb2JhYmlsaXRpZXM6IG5wLm5kYXJyYXksCiAgICAgICAgICAgICAgICAgICAgICBtaW5fbWFyZ2luOiBmbG9hdCA9IDAuMCkgLT4gbnAubmRhcnJheToKICAgICIiIlJlb3JkZXIgb25seSBtb2RlbC12aXNpYmxlIFBPUCBlZGdlcywga2VlcGluZyB1bnNlZW4gcG9zaXRpb25zIGZpeGVkLgoKICAgIENhbmRpZGF0ZSBtZW1iZXJzaGlwIGFuZCBlYWNoIHJvdydzIG9yaWdpbmFsIHVuc2Vlbi1lZGdlIHBvc2l0aW9ucyByZW1haW4KICAgIGV4YWN0bHkgdW5jaGFuZ2VkLiBUaGUgY2FsbGVyIGNhbiBjb21wYXJlIHRoaXMgYXJtIHdpdGggYSBzaGFtIGxpdmUgaG9vay4KICAgICIiIgogICAgcG9wLCBuZWlnaGJvdXJzLCBwcm9iYWJpbGl0aWVzID0gX3ZhbGlkYXRlKHBvcCwgbmVpZ2hib3VycywgcHJvYmFiaWxpdGllcykKICAgIGlmIG5vdCBucC5pc2Zpbml0ZShtaW5fbWFyZ2luKSBvciBtaW5fbWFyZ2luIDwgMDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtaW5fbWFyZ2luIG11c3QgYmUgZmluaXRlIGFuZCBub25uZWdhdGl2ZSIpCiAgICBvdXQgPSBwb3AuY29weSgpCiAgICBmb3Igc2VlZCBpbiByYW5nZShsZW4ocG9wKSk6CiAgICAgICAgc2NvcmVzLCBwcmVzZW50ID0gX3BvcF9zY29yZXMocG9wW3NlZWRdLCBuZWlnaGJvdXJzLCBwcm9iYWJpbGl0aWVzKQogICAgICAgIGZvciBub2RlIGluIHJhbmdlKGxlbihuZWlnaGJvdXJzKSk6CiAgICAgICAgICAgIHNsb3RzID0gbnAuZmxhdG5vbnplcm8ocHJlc2VudFtub2RlXSkKICAgICAgICAgICAgaWYgbGVuKHNsb3RzKSA8IDIgb3IgbnAucHRwKHNjb3Jlc1tub2RlLCBzbG90c10pIDw9IG1pbl9tYXJnaW46CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBvcmRlciA9IG5wLmFyZ3NvcnQoLXNjb3Jlc1tub2RlLCBzbG90c10sIGtpbmQ9InN0YWJsZSIpCiAgICAgICAgICAgIG91dFtzZWVkLCBub2RlLCBzbG90c10gPSBwb3Bbc2VlZCwgbm9kZSwgc2xvdHNbb3JkZXJdXQogICAgcmV0dXJuIG91dAoKCmRlZiBfZWRnZV9rZXlzKG46IGludCwgZGVzdGluYXRpb246IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICBvcmlnaW4gPSBucC5hcmFuZ2UobiwgZHR5cGU9bnAuaW50NjQpWzosIE5vbmVdCiAgICBkZXN0ID0gbnAuYXNhcnJheShkZXN0aW5hdGlvbiwgZHR5cGU9bnAuaW50NjQpCiAgICBrZXlzID0gbnAubWluaW11bShvcmlnaW4sIGRlc3QpICogbnAuaW50NjQobikgKyBucC5tYXhpbXVtKG9yaWdpbiwgZGVzdCkKICAgIHJldHVybiBucC53aGVyZShkZXN0ID49IDAsIGtleXMsIC0xKQoKCmRlZiBwcm9wb3NlX3NwYXJzZV9hZGQocG9wOiBucC5uZGFycmF5LCBuZWlnaGJvdXJzOiBucC5uZGFycmF5LCBwcm9iYWJpbGl0aWVzOiBucC5uZGFycmF5LAogICAgICAgICAgICAgICAgICAgICAgIGNvbmZpZzogU3BhcnNlQWRkQ29uZmlnID0gU3BhcnNlQWRkQ29uZmlnKCkpIC0+IFNwYXJzZUFkZFJlc3VsdDoKICAgICIiIlNlbGVjdCBhdCBtb3N0IG9uZSBnbG9iYWxseSBQT1Atbm92ZWwgZWRnZSBmb3IgYSBzdWJzZXQgb2Ygbm9kZXMuCgogICAgVGhlIG5vLW9wIG9wdGlvbiBpcyBleHBsaWNpdC4gTm92ZWx0eSBpcyBjaGVja2VkIGFnYWluc3QgdGhlIGZ1bGwKICAgICp1bmRpcmVjdGVkKiBQT1AgY2FuZGlkYXRlIGdyYXBoIGZvciBlYWNoIHNlZWQsIG1hdGNoaW5nIHRoZSB2OCBwcm90b2NvbC4KICAgIFNlbGVjdGlvbiBpcyBkZXRlcm1pbmlzdGljOiBwcm9iYWJpbGl0eSByYW5rIGJyZWFrcyB0aWVzIGJ5IG9yaWdpbmFsCiAgICBncmFwaCBvcmRlcjsgbm9kZSBJRCBicmVha3MgZXF1YWwgcHJveHkgc2NvcmVzLgogICAgIiIiCiAgICBwb3AsIG5laWdoYm91cnMsIHByb2JhYmlsaXRpZXMgPSBfdmFsaWRhdGUocG9wLCBuZWlnaGJvdXJzLCBwcm9iYWJpbGl0aWVzKQogICAgc2VlZHMsIG4sIGtfcG9wID0gcG9wLnNoYXBlCiAgICBpZiBjb25maWcubWluX3Njb3JlZF9wb3AgPiBrX3BvcDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJtaW5fc2NvcmVkX3BvcCBleGNlZWRzIFBPUCB3aWR0aCIpCiAgICBkZXN0aW5hdGlvbiA9IG5wLmZ1bGwoKHNlZWRzLCBuKSwgLTEsIGR0eXBlPW5wLmludDMyKQogICAgcHJvcG9zZWQgPSBucC5mdWxsKChzZWVkcywgbiksIC0xLCBkdHlwZT1ucC5pbnQzMikKICAgIHBpY2tlZF9wcm9iID0gbnAuZnVsbCgoc2VlZHMsIG4pLCBucC5uYW4sIGR0eXBlPW5wLmZsb2F0NjQpCiAgICBzY29yZSA9IG5wLmZ1bGwoKHNlZWRzLCBuKSwgbnAubmFuLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgY292ZXJlZCA9IG5wLnplcm9zKChzZWVkcywgbiksIGR0eXBlPW5wLmludDE2KQogICAgZWxpZ2libGUgPSBucC56ZXJvcygoc2VlZHMsIG4pLCBkdHlwZT1ib29sKQogICAgcmFua2VkX2dyYXBoID0gbnAuYXJnc29ydCgtcHJvYmFiaWxpdGllcywgYXhpcz0xLCBraW5kPSJzdGFibGUiKQogICAgZ3JhcGhfa2V5cyA9IF9lZGdlX2tleXMobiwgbmVpZ2hib3VycykKICAgIGxpbWl0ID0gaW50KG5wLmZsb29yKGNvbmZpZy5tYXhfZnJhY3Rpb24gKiBuKSkKCiAgICBmb3Igc2VlZCBpbiByYW5nZShzZWVkcyk6CiAgICAgICAgcG9wX2tleXMgPSBucC51bmlxdWUoX2VkZ2Vfa2V5cyhuLCBwb3Bbc2VlZF0pKQogICAgICAgIHBvcF9rZXlzID0gcG9wX2tleXNbcG9wX2tleXMgPj0gMF0KICAgICAgICBub3ZlbCA9IH5ucC5pc2luKGdyYXBoX2tleXMsIHBvcF9rZXlzKQogICAgICAgIHJhbmtlZF9ub3ZlbCA9IG5wLnRha2VfYWxvbmdfYXhpcyhub3ZlbCwgcmFua2VkX2dyYXBoLCBheGlzPTEpCiAgICAgICAgaGFzX25vdmVsID0gcmFua2VkX25vdmVsLmFueShheGlzPTEpCiAgICAgICAgZmlyc3RfcmFuayA9IHJhbmtlZF9ub3ZlbC5hcmdtYXgoYXhpcz0xKQogICAgICAgIGNob3Nlbl9jb2wgPSByYW5rZWRfZ3JhcGhbbnAuYXJhbmdlKG4pLCBmaXJzdF9yYW5rXQogICAgICAgIHByb3Bvc2VkW3NlZWQsIGhhc19ub3ZlbF0gPSBuZWlnaGJvdXJzW25wLmFyYW5nZShuKSwgY2hvc2VuX2NvbF1baGFzX25vdmVsXQogICAgICAgIHBpY2tlZF9wcm9iW3NlZWQsIGhhc19ub3ZlbF0gPSBwcm9iYWJpbGl0aWVzW25wLmFyYW5nZShuKSwgY2hvc2VuX2NvbF1baGFzX25vdmVsXQoKICAgICAgICBwb3BfcHJvYiwgcHJlc2VudCA9IF9wb3Bfc2NvcmVzKHBvcFtzZWVkXSwgbmVpZ2hib3VycywgcHJvYmFiaWxpdGllcykKICAgICAgICBjb3ZlcmVkW3NlZWRdID0gcHJlc2VudC5zdW0oYXhpcz0xKQogICAgICAgIHdlYWtlc3Rfc2NvcmVkID0gbnAubWluKG5wLndoZXJlKHByZXNlbnQsIHBvcF9wcm9iLCBucC5pbmYpLCBheGlzPTEpCiAgICAgICAgc2NvcmVbc2VlZCwgaGFzX25vdmVsXSA9IHBpY2tlZF9wcm9iW3NlZWQsIGhhc19ub3ZlbF0gLSB3ZWFrZXN0X3Njb3JlZFtoYXNfbm92ZWxdCiAgICAgICAgZWxpZ2libGVbc2VlZF0gPSAoaGFzX25vdmVsICYgKGNvdmVyZWRbc2VlZF0gPj0gY29uZmlnLm1pbl9zY29yZWRfcG9wKQogICAgICAgICAgICAgICAgICAgICAgICAgICYgKHNjb3JlW3NlZWRdID4gY29uZmlnLm1pbl9tYXJnaW4pKQogICAgICAgIGlmIGxpbWl0OgogICAgICAgICAgICBjYW5kaWRhdGVzID0gbnAuZmxhdG5vbnplcm8oZWxpZ2libGVbc2VlZF0pCiAgICAgICAgICAgIG9yZGVyID0gbnAuYXJnc29ydCgtc2NvcmVbc2VlZCwgY2FuZGlkYXRlc10sIGtpbmQ9InN0YWJsZSIpCiAgICAgICAgICAgIHNlbGVjdGVkID0gY2FuZGlkYXRlc1tvcmRlcls6bGltaXRdXQogICAgICAgICAgICBkZXN0aW5hdGlvbltzZWVkLCBzZWxlY3RlZF0gPSBwcm9wb3NlZFtzZWVkLCBzZWxlY3RlZF0KCiAgICByZXR1cm4gU3BhcnNlQWRkUmVzdWx0KGRlc3RpbmF0aW9uLCBwcm9wb3NlZCwgcGlja2VkX3Byb2IsIHNjb3JlLCBjb3ZlcmVkLCBlbGlnaWJsZSkK', 'v9_live/live_pilot.py': 'IiIiQmV0YVRvdXIgdjkgbGl2ZS1MS0ggcGlsb3QgaGVscGVycy4KClRoaXMgbW9kdWxlIGRlZmluZXMgaW50ZXJ2ZW50aW9ucyBhbmQgdmFsaWRhdGVzIHRoZWlyIGFjdHVhbCB0b3VyIGVuZHBvaW50cy4KVGhlIGZyb3plbiBUb3VyRm9sZCBjaGVja3BvaW50IHN1cHBsaWVzIHJlZmVyZW5jZS1lZGdlIHNjb3Jlczsgd2hldGhlciB0aGVzZQppbnRlcnZlbnRpb25zIGltcHJvdmUgTEtIIGlzIGFuIGVtcGlyaWNhbCBxdWVzdGlvbiBmb3IgdGhlIHBpbG90LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBoYXNobGliCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gYWRhcHRpdmVfcG9saWN5IGltcG9ydCBTcGFyc2VBZGRDb25maWcsIHByb3Bvc2Vfc3BhcnNlX2FkZCwgcmVyYW5rX3Njb3JlZF9wb3AKZnJvbSBjb29yZGluYXRvciBpbXBvcnQgU25hcHNob3QsIG5vb3BfYWN0aW9ucywgcmVhZF90b3VyLCBydW5fc2l4CgoKTElWRV9BUk1TID0gKCJzaGFtIiwgImJldGFfcHJvbW90ZSIsICJiZXRhX3NwYXJzZSIsICJmcmVxdWVuY3lfc2FtZV9nYXRlIikKTkFUSVZFX0FSTVMgPSAoIm5hdGl2ZV9wb3A1IiwgIm5hdGl2ZV9wb3A2IikKCgpkZWYgc2hhMjU2X2ZpbGUocGF0aDogUGF0aCkgLT4gc3RyOgogICAgZGlnZXN0ID0gaGFzaGxpYi5zaGEyNTYoKQogICAgd2l0aCBwYXRoLm9wZW4oInJiIikgYXMgc291cmNlOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogc291cmNlLnJlYWQoNCAqIDEwMjQgKiAxMDI0KSwgYiIiKToKICAgICAgICAgICAgZGlnZXN0LnVwZGF0ZShjaHVuaykKICAgIHJldHVybiBkaWdlc3QuaGV4ZGlnZXN0KCkKCgpkZWYgY2FuZGlkYXRlX2FycmF5KHNuYXBzaG90czogU2VxdWVuY2VbU25hcHNob3RdLCBmaWVsZDogc3RyKSAtPiBucC5uZGFycmF5OgogICAgcm93cyA9IFtnZXRhdHRyKHNuYXBzaG90LCBmaWVsZCkgZm9yIHNuYXBzaG90IGluIHNuYXBzaG90c10KICAgIG4gPSBsZW4ocm93c1swXSkKICAgIGlmIGFueShsZW4oaXRlbSkgIT0gbiBmb3IgaXRlbSBpbiByb3dzKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5kaWRhdGUgc25hcHNob3RzIGhhdmUgZGlmZmVyZW50IGRpbWVuc2lvbnMiKQogICAgd2lkdGggPSBtYXgobGVuKHJvdykgZm9yIHNlZWRfcm93cyBpbiByb3dzIGZvciByb3cgaW4gc2VlZF9yb3dzKQogICAgaWYgd2lkdGggPCAxIG9yIHdpZHRoID49IG46CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIkludmFsaWQgY2FuZGlkYXRlIHdpZHRoIHt3aWR0aH0iKQogICAgb3V0ID0gbnAuZnVsbCgobGVuKHJvd3MpLCBuLCB3aWR0aCksIC0xLCBkdHlwZT1ucC5pbnQzMikKICAgIGZvciBtLCBzZWVkX3Jvd3MgaW4gZW51bWVyYXRlKHJvd3MpOgogICAgICAgIGZvciBpLCByb3cgaW4gZW51bWVyYXRlKHNlZWRfcm93cyk6CiAgICAgICAgICAgIGlmIHJvdzoKICAgICAgICAgICAgICAgIG91dFttLCBpLCA6bGVuKHJvdyldID0gW3RvIC0gMSBmb3IgdG8sIF8gaW4gcm93XQogICAgcmV0dXJuIG91dAoKCmRlZiBmaXJzdF90b3VyX2FycmF5cyhzbmFwc2hvdHM6IFNlcXVlbmNlW1NuYXBzaG90XSwgeHk6IG5wLm5kYXJyYXksIFQpIC0+IHR1cGxlW25wLm5kYXJyYXksIG5wLm5kYXJyYXldOgogICAgc3VjY2Vzc29ycywgY29zdHMgPSBbXSwgW10KICAgIGZvciBzbmFwc2hvdCBpbiBzbmFwc2hvdHM6CiAgICAgICAgdG91ciA9IG5wLmFzYXJyYXkoc25hcHNob3QuZmlyc3RfdG91ciwgZHR5cGU9bnAuaW50MzIpIC0gMQogICAgICAgIGlmIG5vdCBULmlzX3ZhbGlkX3RvdXIodG91ciwgbGVuKHh5KSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbnZhbGlkIGZpcnN0IHRvdXIgZnJvbSBzZWVkIHtzbmFwc2hvdC5zZWVkfSIpCiAgICAgICAgc3VjY2Vzc29ycy5hcHBlbmQoVC50b3VyX3N1Y2NfcHJlZCh0b3VyKVswXSkKICAgICAgICBjb3N0cy5hcHBlbmQoVC50b3VyX2xlbmd0aCh4eSwgdG91ciwgIkVVQ18yRCIpKQogICAgcmV0dXJuIG5wLnN0YWNrKHN1Y2Nlc3NvcnMsIGF4aXM9MSkuYXN0eXBlKG5wLmludDMyKSwgbnAuYXNhcnJheShjb3N0cywgZHR5cGU9bnAuZmxvYXQ2NCkKCgpkZWYgX3NhdmVfbnB6X2F0b21pYyhwYXRoOiBQYXRoLCAqKmFycmF5cykgLT4gTm9uZToKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHRtcCA9IHBhdGgud2l0aF9uYW1lKHBhdGgubmFtZSArICIudG1wLm5weiIpCiAgICBucC5zYXZlel9jb21wcmVzc2VkKHRtcCwgKiphcnJheXMpCiAgICBvcy5yZXBsYWNlKHRtcCwgcGF0aCkKCgpkZWYgX2FjdGlvbnNfZnJvbV9kZXN0aW5hdGlvbnMoZGVzdGluYXRpb25zOiBucC5uZGFycmF5LCBzbG90OiBpbnQsIGFscGhhOiBpbnQpIC0+IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dOgogICAgcmV0dXJuIFtbKGludCh0bykgKyAxLCBzbG90LCBhbHBoYSkgaWYgdG8gPj0gMCBlbHNlICgwLCAwLCA1MDApCiAgICAgICAgICAgICBmb3IgdG8gaW4gcm93XSBmb3Igcm93IGluIGRlc3RpbmF0aW9uc10KCgpkZWYgbWFrZV9hY3Rpb25fY2FsbGJhY2soa2luZDogc3RyLCB4eTogbnAubmRhcnJheSwgbW9kZWwsIGRldmljZSwgVCwgTSwKICAgICAgICAgICAgICAgICAgICAgICAgIGF1ZGl0X2RpcjogUGF0aCwgZ3JhcGhfazogaW50ID0gMTYsCiAgICAgICAgICAgICAgICAgICAgICAgICBzcGFyc2VfZnJhY3Rpb246IGZsb2F0ID0gMC4yNSk6CiAgICBpZiBraW5kIG5vdCBpbiBMSVZFX0FSTVM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlVua25vd24gbGl2ZSBhcm0ge2tpbmR9IikKICAgIGF1ZGl0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgZGVmIGNhbGxiYWNrKHNuYXBzaG90czogbGlzdFtTbmFwc2hvdF0pIC0+IGxpc3RbbGlzdFt0dXBsZVtpbnQsIGludCwgaW50XV1dOgogICAgICAgIGlmIGxlbihzbmFwc2hvdHMpICE9IDY6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlRvdXJGb2xkIHJlcXVpcmVzIHNpeCBmaXJzdCB0b3VycyIpCiAgICAgICAgc3VjYywgY29zdHMgPSBmaXJzdF90b3VyX2FycmF5cyhzbmFwc2hvdHMsIHh5LCBUKQogICAgICAgIGxpdmUgPSBjYW5kaWRhdGVfYXJyYXkoc25hcHNob3RzLCAibGl2ZV9jYW5kaWRhdGVzIikKICAgICAgICBuYXRpdmUgPSBjYW5kaWRhdGVfYXJyYXkoc25hcHNob3RzLCAibmF0aXZlX2NhbmRpZGF0ZXMiKQogICAgICAgIHByZWRpY3Rpb24gPSBNLnByZWRpY3RfdG91cmZvbGQobW9kZWwsIHh5LCBncmFwaF9rLCBkZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBlbnNfc3VjYz1zdWNjLCBlbnNfY29zdD1jb3N0cykKICAgICAgICBncmFwaCA9IG5wLmFzYXJyYXkocHJlZGljdGlvblsiTkIiXSwgZHR5cGU9bnAuaW50MzIpCiAgICAgICAgcHJvYiA9IG5wLmFzYXJyYXkocHJlZGljdGlvblsicHJvYiJdLCBkdHlwZT1ucC5mbG9hdDMyKQogICAgICAgIGlmIGtpbmQgPT0gInNoYW0iOgogICAgICAgICAgICBkZXN0aW5hdGlvbnMgPSBucC5mdWxsKCg2LCBsZW4oeHkpKSwgLTEsIGR0eXBlPW5wLmludDMyKQogICAgICAgICAgICBhY3Rpb25zID0gW25vb3BfYWN0aW9ucyhsZW4oeHkpKSBmb3IgXyBpbiByYW5nZSg2KV0KICAgICAgICBlbGlmIGtpbmQgPT0gImJldGFfcHJvbW90ZSI6CiAgICAgICAgICAgIHJlb3JkZXJlZCA9IHJlcmFua19zY29yZWRfcG9wKGxpdmUsIGdyYXBoLCBwcm9iKQogICAgICAgICAgICBkZXN0aW5hdGlvbnMgPSBucC53aGVyZShyZW9yZGVyZWRbOiwgOiwgMF0gIT0gbGl2ZVs6LCA6LCAwXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcmVvcmRlcmVkWzosIDosIDBdLCAtMSkuYXN0eXBlKG5wLmludDMyKQogICAgICAgICAgICBhY3Rpb25zID0gX2FjdGlvbnNfZnJvbV9kZXN0aW5hdGlvbnMoZGVzdGluYXRpb25zLCBzbG90PTEsIGFscGhhPTApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc3BhcnNlID0gcHJvcG9zZV9zcGFyc2VfYWRkKAogICAgICAgICAgICAgICAgbGl2ZSwgZ3JhcGgsIHByb2IsCiAgICAgICAgICAgICAgICBTcGFyc2VBZGRDb25maWcobWF4X2ZyYWN0aW9uPXNwYXJzZV9mcmFjdGlvbiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtaW5fc2NvcmVkX3BvcD0yLCBtaW5fbWFyZ2luPTAuMCksCiAgICAgICAgICAgICkKICAgICAgICAgICAgZGVzdGluYXRpb25zID0gc3BhcnNlLmRlc3RpbmF0aW9uLmNvcHkoKQogICAgICAgICAgICBpZiBraW5kID09ICJmcmVxdWVuY3lfc2FtZV9nYXRlIjoKICAgICAgICAgICAgICAgICMgS2VlcCBCZXRhJ3Mgbm9kZS1sZXZlbCBnYXRlIGFuZCBpbnRlcnZlbnRpb24gY291bnQsIHdoaWxlCiAgICAgICAgICAgICAgICAjIHJlcGxhY2luZyBpdHMgZWRnZSBjaG9pY2Ugd2l0aCBlbnNlbWJsZSBmcmVxdWVuY3kuCiAgICAgICAgICAgICAgICBmcmVxdWVuY3kgPSBULmVuc2VtYmxlX2ZyZXF1ZW5jeShncmFwaCwgc3VjYykKICAgICAgICAgICAgICAgIGZyZXF1ZW5jeV9yYW5rID0gVC5mcmVxdWVuY3lfY2FuZGlkYXRlcygKICAgICAgICAgICAgICAgICAgICBncmFwaCwgcHJlZGljdGlvblsiRCJdLCBmcmVxdWVuY3ksIGs9Z3JhcGguc2hhcGVbMV0pCiAgICAgICAgICAgICAgICBuID0gbGVuKHh5KQogICAgICAgICAgICAgICAgZm9yIHNlZWQgaW4gcmFuZ2UoNik6CiAgICAgICAgICAgICAgICAgICAgZXhpc3RpbmcgPSB7CiAgICAgICAgICAgICAgICAgICAgICAgIChtaW4oaSwgaW50KHYpKSwgbWF4KGksIGludCh2KSkpCiAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpLCByb3cgaW4gZW51bWVyYXRlKGxpdmVbc2VlZF0pIGZvciB2IGluIHJvdyBpZiB2ID49IDAKICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICAgICAgZm9yIGkgaW4gbnAuZmxhdG5vbnplcm8oZGVzdGluYXRpb25zW3NlZWRdID49IDApOgogICAgICAgICAgICAgICAgICAgICAgICBjaG9pY2UgPSAtMQogICAgICAgICAgICAgICAgICAgICAgICBmb3IgdiBpbiBmcmVxdWVuY3lfcmFua1tpXToKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHYgPSBpbnQodikKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGlmIChtaW4oaSwgdiksIG1heChpLCB2KSkgbm90IGluIGV4aXN0aW5nOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNob2ljZSA9IHYKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgICAgICAgICBkZXN0aW5hdGlvbnNbc2VlZCwgaV0gPSBjaG9pY2UKICAgICAgICAgICAgIyBTbG90IDMgdGVzdHMgd2hldGhlciBhIHNwYXJzZSBwcm9wb3NhbCBjYW4gYmUgc2VhcmNoZWQgZWFybHkKICAgICAgICAgICAgIyB3aGlsZSByZXRhaW5pbmcgZXZlcnkgbmF0aXZlIGNhbmRpZGF0ZSBlZGdlIGluIHRoZSBsaXZlIHNldC4KICAgICAgICAgICAgYWN0aW9ucyA9IF9hY3Rpb25zX2Zyb21fZGVzdGluYXRpb25zKGRlc3RpbmF0aW9ucywgc2xvdD0zLCBhbHBoYT01MDApCiAgICAgICAgX3NhdmVfbnB6X2F0b21pYyhhdWRpdF9kaXIgLyAicG9saWN5X2FycmF5cy5ucHoiLCBmaXJzdF9zdWNjZXNzb3JzPXN1Y2MsCiAgICAgICAgICAgICAgICAgICAgICAgICBmaXJzdF9jb3N0cz1jb3N0cywgbmF0aXZlX2NhbmRpZGF0ZXM9bmF0aXZlLAogICAgICAgICAgICAgICAgICAgICAgICAgbGl2ZV9jYW5kaWRhdGVzPWxpdmUsIG1vZGVsX2dyYXBoPWdyYXBoLAogICAgICAgICAgICAgICAgICAgICAgICAgbW9kZWxfcHJvYmFiaWxpdHk9cHJvYiwgY2hvc2VuX2Rlc3RpbmF0aW9ucz1kZXN0aW5hdGlvbnMpCiAgICAgICAgKGF1ZGl0X2RpciAvICJwb2xpY3lfbWV0YS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAgICAgImFybSI6IGtpbmQsCiAgICAgICAgICAgICJhY3Rpb25zX3Blcl9zZWVkIjogW2ludChucC5zdW0ocm93ID49IDApKSBmb3Igcm93IGluIGRlc3RpbmF0aW9uc10sCiAgICAgICAgICAgICJtb2RlbF9pbmZlcmVuY2VfcyI6IGZsb2F0KHByZWRpY3Rpb25bInRfdG90YWwiXSksCiAgICAgICAgICAgICJzcGFyc2VfZnJhY3Rpb24iOiBzcGFyc2VfZnJhY3Rpb24gaWYga2luZCBpbiAoImJldGFfc3BhcnNlIiwgImZyZXF1ZW5jeV9zYW1lX2dhdGUiKSBlbHNlIE5vbmUsCiAgICAgICAgfSwgaW5kZW50PTIpICsgIlxuIikKICAgICAgICByZXR1cm4gYWN0aW9ucwoKICAgIHJldHVybiBjYWxsYmFjawoKCmRlZiBfcmVhZF9iZXN0KHBhdGg6IFBhdGgsIHh5OiBucC5uZGFycmF5LCBUKSAtPiBpbnQ6CiAgICAjIFRoZSBob29rIHdyaXRlcyBhIGNvbXBhY3QgYXRvbWljIGludGVnZXIgbGlzdDsgbmF0aXZlIExLSCB3cml0ZXMgYQogICAgIyBUU1BMSUIgVE9VUl9TRUNUSU9OIGZpbGUuICBSZWNvbXB1dGUgY29zdHMgZnJvbSBlaXRoZXIgZXhhY3QgdG91ci4KICAgIGZpcnN0X3Rva2VuID0gcGF0aC5yZWFkX3RleHQoKS5zcGxpdChtYXhzcGxpdD0xKVswXQogICAgaWYgZmlyc3RfdG9rZW4uaXNkaWdpdCgpOgogICAgICAgIHRvdXIgPSBucC5hc2FycmF5KHJlYWRfdG91cihwYXRoKSwgZHR5cGU9bnAuaW50MzIpIC0gMQogICAgZWxzZToKICAgICAgICB0b3VyID0gVC5yZWFkX3RvdXIocGF0aCkuYXN0eXBlKG5wLmludDMyKQogICAgaWYgbm90IFQuaXNfdmFsaWRfdG91cih0b3VyLCBsZW4oeHkpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBiZXN0IHRvdXIge3BhdGh9IikKICAgIHJldHVybiBpbnQoVC50b3VyX2xlbmd0aCh4eSwgdG91ciwgIkVVQ18yRCIpKQoKCmRlZiBydW5fbGl2ZV9hcm0oa2luZDogc3RyLCBwYXRjaGVkX2JpbmFyeTogUGF0aCwgcHJvYmxlbTogUGF0aCwgeHk6IG5wLm5kYXJyYXksCiAgICAgICAgICAgICAgICAgd29ya19kaXI6IFBhdGgsIHNlZWRzOiBTZXF1ZW5jZVtpbnRdLCBidWRnZXRfczogZmxvYXQsCiAgICAgICAgICAgICAgICAgbW9kZWwsIGRldmljZSwgVCwgTSwgKiwgbWF4X3RyaWFsczogaW50ID0gMTAwXzAwMF8wMDAsCiAgICAgICAgICAgICAgICAgc3BhcnNlX2ZyYWN0aW9uOiBmbG9hdCA9IDAuMjUpIC0+IGRpY3Q6CiAgICBjYWxsYmFjayA9IG1ha2VfYWN0aW9uX2NhbGxiYWNrKGtpbmQsIHh5LCBtb2RlbCwgZGV2aWNlLCBULCBNLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB3b3JrX2Rpci5wYXJlbnQgLyAod29ya19kaXIubmFtZSArICJfYXVkaXQiKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3BhcnNlX2ZyYWN0aW9uPXNwYXJzZV9mcmFjdGlvbikKICAgIHJlc3VsdCA9IHJ1bl9zaXgocGF0Y2hlZF9iaW5hcnksIHByb2JsZW0sIHdvcmtfZGlyLCBzZWVkcywgYnVkZ2V0X3MsCiAgICAgICAgICAgICAgICAgICAgIGNhbGxiYWNrLCBtYXhfdHJpYWxzPW1heF90cmlhbHMpCiAgICBjb3N0cyA9IFtdCiAgICBmb3IgcnVuIGluIHJlc3VsdFsicnVucyJdOgogICAgICAgIGlmIHJ1blsiYmVzdF90b3VyIl0gaXMgTm9uZToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTm8gYmVzdCB0b3VyIGluIHtraW5kfSwgc2VlZCB7cnVuWydzZWVkJ119IikKICAgICAgICBjb3N0cy5hcHBlbmQoX3JlYWRfYmVzdChQYXRoKHJ1blsiYmVzdF90b3VyIl0pLCB4eSwgVCkpCiAgICBpZiBub3QgcmVzdWx0WyJiYXJyaWVyX2NvbXBsZXRlZCJdOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkZpcnN0LXRvdXIgYmFycmllciBpbmNvbXBsZXRlIGluIHtraW5kfSIpCiAgICBpZiBhbnkocnVuWyJhcHBsaWVkX3N0YXRzIl0gaXMgTm9uZSBmb3IgcnVuIGluIHJlc3VsdFsicnVucyJdKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJPbmUgb3IgbW9yZSBob29rIGFjdGlvbnMgd2VyZSBub3QgYXBwbGllZCBpbiB7a2luZH0iKQogICAgcmVzdWx0LnVwZGF0ZShhcm09a2luZCwgc2VlZF9jb3N0cz1jb3N0cywgYmVzdF9jb3N0PW1pbihjb3N0cyksCiAgICAgICAgICAgICAgICAgIHBvbGljeV9tZXRhPXN0cih3b3JrX2Rpci5wYXJlbnQgLyAod29ya19kaXIubmFtZSArICJfYXVkaXQiKSksCiAgICAgICAgICAgICAgICAgIGJpbmFyeV9zaGEyNTY9c2hhMjU2X2ZpbGUocGF0Y2hlZF9iaW5hcnkpKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBydW5fbmF0aXZlX2FybShraW5kOiBzdHIsIG5hdGl2ZV9iaW5hcnk6IFBhdGgsIHByb2JsZW06IFBhdGgsIHh5OiBucC5uZGFycmF5LAogICAgICAgICAgICAgICAgICAgd29ya19kaXI6IFBhdGgsIHNlZWRzOiBTZXF1ZW5jZVtpbnRdLCBidWRnZXRfczogZmxvYXQsIFQsCiAgICAgICAgICAgICAgICAgICAqLCBtYXhfdHJpYWxzOiBpbnQgPSAxMDBfMDAwXzAwMCkgLT4gZGljdDoKICAgIGlmIGtpbmQgbm90IGluIE5BVElWRV9BUk1TIG9yIGxlbihzZWVkcykgIT0gNiBvciB3b3JrX2Rpci5leGlzdHMoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJJbnZhbGlkIG5hdGl2ZSBhcm0sIHNlZWQgcGFuZWwsIG9yIHdvcmsgZGlyZWN0b3J5IikKICAgIHdvcmtfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSkKICAgIGsgPSA1IGlmIGtpbmQgPT0gIm5hdGl2ZV9wb3A1IiBlbHNlIDYKICAgIHByb2Nlc3Nlcywgc3RyZWFtcyA9IFtdLCBbXQogICAgc3RhcnQgPSB0aW1lLm1vbm90b25pYygpCiAgICBkZWFkbGluZSA9IHN0YXJ0ICsgYnVkZ2V0X3MKICAgIHRyeToKICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgYmFzZSA9IHdvcmtfZGlyIC8gZiJzZWVkX3tzZWVkfSIKICAgICAgICAgICAgYmFzZS5ta2RpcigpCiAgICAgICAgICAgIHBhcmFtcyA9IHsKICAgICAgICAgICAgICAgICJQUk9CTEVNX0ZJTEUiOiBzdHIocHJvYmxlbS5yZXNvbHZlKCkpLAogICAgICAgICAgICAgICAgIkNBTkRJREFURV9TRVRfVFlQRSI6ICJQT1BNVVNJQyIsICJTVUJHUkFESUVOVCI6ICJOTyIsCiAgICAgICAgICAgICAgICAiTUFYX0NBTkRJREFURVMiOiBrLCAiU0VFRCI6IHNlZWQsICJSVU5TIjogMSwKICAgICAgICAgICAgICAgICJNQVhfVFJJQUxTIjogbWF4X3RyaWFscywgIlRJTUVfTElNSVQiOiBtYXgoMTAuMCwgMTAuMCAqIGJ1ZGdldF9zKSwKICAgICAgICAgICAgICAgICJUUkFDRV9MRVZFTCI6IDAsICJTVE9QX0FUX09QVElNVU0iOiAiTk8iLAogICAgICAgICAgICAgICAgIk9VVFBVVF9UT1VSX0ZJTEUiOiBzdHIoYmFzZSAvICJiZXN0LnRvdXIiKSwKICAgICAgICAgICAgfQogICAgICAgICAgICBwYXIgPSBiYXNlIC8gInJ1bi5wYXIiCiAgICAgICAgICAgIFQud3JpdGVfcGFyKHBhciwgcGFyYW1zKQogICAgICAgICAgICBzdGRvdXQgPSAoYmFzZSAvICJzdGRvdXQudHh0Iikub3BlbigidyIpCiAgICAgICAgICAgIHN0ZGVyciA9IChiYXNlIC8gInN0ZGVyci50eHQiKS5vcGVuKCJ3IikKICAgICAgICAgICAgc3RyZWFtcy5leHRlbmQoKHN0ZG91dCwgc3RkZXJyKSkKICAgICAgICAgICAgZW52ID0gZGljdChvcy5lbnZpcm9uKQogICAgICAgICAgICBlbnYucG9wKCJCRVRBVE9VUl9IT09LX0RJUiIsIE5vbmUpCiAgICAgICAgICAgIHByb2Nlc3Nlcy5hcHBlbmQoc3VicHJvY2Vzcy5Qb3BlbigKICAgICAgICAgICAgICAgIFtzdHIobmF0aXZlX2JpbmFyeS5yZXNvbHZlKCkpLCBzdHIocGFyLnJlc29sdmUoKSldLCBjd2Q9YmFzZSwKICAgICAgICAgICAgICAgIHN0ZG91dD1zdGRvdXQsIHN0ZGVycj1zdGRlcnIsIGVudj1lbnYpKQogICAgICAgIHdoaWxlIGFueShwcm9jZXNzLnBvbGwoKSBpcyBOb25lIGZvciBwcm9jZXNzIGluIHByb2Nlc3NlcykgYW5kIHRpbWUubW9ub3RvbmljKCkgPCBkZWFkbGluZToKICAgICAgICAgICAgdGltZS5zbGVlcCgwLjAyKQogICAgZmluYWxseToKICAgICAgICBmb3IgcHJvY2VzcyBpbiBwcm9jZXNzZXM6CiAgICAgICAgICAgIGlmIHByb2Nlc3MucG9sbCgpIGlzIE5vbmU6CiAgICAgICAgICAgICAgICBwcm9jZXNzLnRlcm1pbmF0ZSgpCiAgICAgICAgZm9yIHByb2Nlc3MgaW4gcHJvY2Vzc2VzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBwcm9jZXNzLndhaXQodGltZW91dD0yKQogICAgICAgICAgICBleGNlcHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZDoKICAgICAgICAgICAgICAgIHByb2Nlc3Mua2lsbCgpCiAgICAgICAgICAgICAgICBwcm9jZXNzLndhaXQoKQogICAgICAgIGZvciBzdHJlYW0gaW4gc3RyZWFtczoKICAgICAgICAgICAgc3RyZWFtLmNsb3NlKCkKICAgIGNvc3RzID0gW10KICAgIHJ1bnMgPSBbXQogICAgZm9yIHNlZWQsIHByb2Nlc3MgaW4gemlwKHNlZWRzLCBwcm9jZXNzZXMpOgogICAgICAgIGJlc3QgPSB3b3JrX2RpciAvIGYic2VlZF97c2VlZH0iIC8gImJlc3QudG91ciIKICAgICAgICBpZiBub3QgYmVzdC5leGlzdHMoKToKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKGYiTm8gYmVzdCB0b3VyIGluIHtraW5kfSwgc2VlZCB7c2VlZH0iKQogICAgICAgIGNvc3QgPSBfcmVhZF9iZXN0KGJlc3QsIHh5LCBUKQogICAgICAgIGNvc3RzLmFwcGVuZChjb3N0KQogICAgICAgIHJ1bnMuYXBwZW5kKHsic2VlZCI6IHNlZWQsICJiZXN0X3RvdXIiOiBzdHIoYmVzdCksCiAgICAgICAgICAgICAgICAgICAgICJyZXR1cm5jb2RlIjogcHJvY2Vzcy5yZXR1cm5jb2RlLCAiY29zdCI6IGNvc3R9KQogICAgcmVzdWx0ID0geyJhcm0iOiBraW5kLCAic2VlZHMiOiBsaXN0KHNlZWRzKSwgIndhbGxfYnVkZ2V0X3MiOiBidWRnZXRfcywKICAgICAgICAgICAgICAid2FsbF9lbGFwc2VkX3MiOiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnQsICJzZWVkX2Nvc3RzIjogY29zdHMsCiAgICAgICAgICAgICAgImJlc3RfY29zdCI6IG1pbihjb3N0cyksICJydW5zIjogcnVucywKICAgICAgICAgICAgICAiYmluYXJ5X3NoYTI1NiI6IHNoYTI1Nl9maWxlKG5hdGl2ZV9iaW5hcnkpfQogICAgKHdvcmtfZGlyIC8gInJ1bl9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVzdWx0LCBpbmRlbnQ9MikgKyAiXG4iKQogICAgcmV0dXJuIHJlc3VsdAo='}
CODE_ROOT = WORK_ROOT / "code"
for rel, encoded in BUNDLED_FILES.items():
    dest = CODE_ROOT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    content = base64.b64decode(encoded)
    if dest.exists():
        assert dest.read_bytes() == content, f"Kod dosyası değişti: {rel}"
    else:
        dest.write_bytes(content)
    drive_dest = OUTPUT_ROOT / "code" / rel
    drive_dest.parent.mkdir(parents=True, exist_ok=True)
    if drive_dest.exists():
        assert drive_dest.read_bytes() == content, f"Drive kod dosyası değişti: {rel}"
    else:
        drive_dest.write_bytes(content)
CODE_SHA256 = {rel: hashlib.sha256(base64.b64decode(encoded)).hexdigest()
               for rel, encoded in BUNDLED_FILES.items()}
sys.path[:0] = [str(CODE_ROOT / "livehook"), str(CODE_ROOT / "policy"), str(CODE_ROOT / "v9_live")]
import build_livehook
from coordinator import read_tour
from live_pilot import run_live_arm, run_native_arm, sha256_file
print("Deney kodu doğrulandı:", CODE_SHA256)


Deney kodu doğrulandı: {'livehook/BetaTourHook.h': '078b46b97517749c73144a3a272b9680436578e9beb897687d5d824fab842cf7', 'livehook/build_livehook.py': '1c1d86cae6b9fede0458d38b969a02cd49d0ff3fa20470288e877c2b7e078a82', 'livehook/coordinator.py': '15ff347715036a05991a5fb1cef9b188cfffe10d600f7810f8cd9bb98ae909ba', 'policy/adaptive_policy.py': '1646995c3075d60088e0a73c42044ad6065987028e48193780ac47d7e77075b8', 'v9_live/live_pilot.py': '2cd13337ef7c9c84f18b75c01bc98738fef8cb8c433416b43eaad4a56fca1000'}


In [ ]:
LKH_ROOT = WORK_ROOT / "lkh"
LKH_ROOT.mkdir(exist_ok=True)
SOURCE_DIR = LKH_ROOT / "LKH-3.0.14"
SOURCE_INFO_PATH = LKH_ROOT / "source_info.json"
if not SOURCE_DIR.exists():
    sources = [
        ("official", "https://webhotel4.ruc.dk/~keld/research/LKH-3/LKH-3.0.14.tgz", None, 12),
        ("pinned_mirror", "https://raw.githubusercontent.com/ZhekaiChen/ALCO/main/third_party/LKH3/LKH-3.0.14.tar", LKH_TAR_SHA256, 90),
    ]
    last_error = None
    for label, url, expected_sha, timeout_s in sources:
        archive = LKH_ROOT / ("LKH-3.0.14.tgz" if url.endswith(".tgz") else "LKH-3.0.14.tar")
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=timeout_s) as response, archive.open("wb") as out:
                shutil.copyfileobj(response, out)
            digest = hashlib.sha256(archive.read_bytes()).hexdigest()
            if expected_sha:
                assert digest == expected_sha, f"LKH arşiv SHA256 uyuşmuyor: {digest}"
            with tarfile.open(archive, "r:*") as tar:
                try:
                    tar.extractall(LKH_ROOT, filter="data")
                except TypeError:
                    tar.extractall(LKH_ROOT)
            assert SOURCE_DIR.exists()
            actual_source_sha = hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest()
            assert actual_source_sha == LKH_FINDTOUR_SHA256, f"LKH 3.0.14 kaynağı beklenenden farklı: {actual_source_sha}"
            SOURCE_INFO_PATH.write_text(json.dumps({"label": label, "url": url, "archive_sha256": digest}, indent=2))
            break
        except Exception as exc:
            last_error = exc
            if SOURCE_DIR.exists():
                shutil.rmtree(SOURCE_DIR)
    else:
        raise RuntimeError(f"LKH indirilemedi: {last_error}")
assert hashlib.sha256((SOURCE_DIR / "SRC/FindTour.c").read_bytes()).hexdigest() == LKH_FINDTOUR_SHA256
if not SOURCE_INFO_PATH.exists():
    cached = next((p for p in (LKH_ROOT / "LKH-3.0.14.tar", LKH_ROOT / "LKH-3.0.14.tgz") if p.exists()), None)
    assert cached is not None, "LKH kaynak kayıt dosyası eksik; kaynak arşiv bulunamadı"
    SOURCE_INFO_PATH.write_text(json.dumps({"label": "recovered_cache", "url": "local_cache",
                                            "archive_sha256": hashlib.sha256(cached.read_bytes()).hexdigest()}, indent=2))
NATIVE_LKH = SOURCE_DIR / "LKH"
if not NATIVE_LKH.exists():
    with (LKH_ROOT / "native_build.log").open("w") as build_log:
        subprocess.run(["make", "-j", str(min(6, CPU_COUNT))], cwd=SOURCE_DIR, check=True,
                       stdout=build_log, stderr=subprocess.STDOUT)
PATCHED_DIR = LKH_ROOT / "LKH-3.0.14-BetaTour"
PATCHED_LKH = PATCHED_DIR / "LKH"
if not PATCHED_LKH.exists():
    if PATCHED_DIR.exists():
        shutil.rmtree(PATCHED_DIR)
    build_livehook.build(SOURCE_DIR, PATCHED_DIR, jobs=min(6, CPU_COUNT))
BUILD_INFO = {"native_sha256": sha256_file(NATIVE_LKH),
              "patched_sha256": sha256_file(PATCHED_LKH),
              "upstream_findtour_sha256": LKH_FINDTOUR_SHA256}
origin_path = OUTPUT_ROOT / "code" / "lkh_source_info.json"
if not origin_path.exists():
    origin_path.write_text(SOURCE_INFO_PATH.read_text())
print("LKH 3.0.14 derlendi:", BUILD_INFO)


LKH 3.0.14 derlendi: {'native_sha256': '47a26176a73cb8b361b0a4e5be1f3f2e4f41a5167f83426c5cbf7371e46ac585', 'patched_sha256': '3c6e8eee7d4018e92db3b8fb171fb4981f8863a9837091cbb89b677ba9ca58e1', 'upstream_findtour_sha256': '567e72acc9b7392488dd158be4df0267560424204a0e88d2d9b62f923a7c0eee'}


In [ ]:
def write_json_atomic(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
    os.replace(tmp, path)

MANIFEST = {"schema": 1, "profile": "pilot_old_v8_instances",
            "experiment_name": EXPERIMENT_NAME, "arms": ARMS, "seeds": SEEDS,
            "instance_specs": INSTANCE_SPECS, "xy_hashes": EXPECTED_XY_SHA256,
            "sparse_fraction": SPARSE_FRACTION, "repo_commit": REPO_COMMIT,
            "checkpoint_sha256": CHECKPOINT_SHA256, "hardware": HARDWARE,
            "code_sha256": CODE_SHA256, "build": BUILD_INFO,
            "analysis_plan": "Pilot only; compare endpoints descriptively and choose one frozen policy before a new holdout."}
manifest_path = OUTPUT_ROOT / "manifest.json"
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text())
    for key in MANIFEST:
        if key != "hardware":
            assert saved[key] == MANIFEST[key], f"Drive'daki pilotun {key} ayarı değişti. Yeni EXPERIMENT_NAME kullanın."
    for key in ("gpu", "vcpus"):
        assert saved["hardware"][key] == HARDWARE[key], f"Donanım {key} değişti; eşli pilot için aynı türü seçin."
    if saved["hardware"] != HARDWARE:
        print("UYARI: CPU modeli veya kütüphane sürümü değişti; kol donanımları ayrı kaydedilecek.")
else:
    write_json_atomic(manifest_path, MANIFEST)

# This warms CUDA kernels and verifies the first six tours against native LKH.
warm_xy = T.to_int_coords(T.gen_instance(100, "uniform", np.random.default_rng(9090)))
warm_succ = np.stack([np.roll(np.arange(100), -1)] * 6, axis=1).astype(np.int32)
M.predict_tourfold(MODEL, warm_xy, 16, DEVICE, ens_succ=warm_succ, ens_cost=np.ones(6))
warm_problem = WORK_ROOT / "warmup.tsp"
T.write_tsp(warm_problem, warm_xy)
warm_dir = WORK_ROOT / "warmup" / uuid.uuid4().hex
warm_dir.parent.mkdir(exist_ok=True)
warm = run_live_arm("sham", PATCHED_LKH, warm_problem, warm_xy, warm_dir,
                    SEEDS, 15.0, MODEL, DEVICE, T, M, max_trials=2)
def edge_set(tour):
    tour = np.asarray(tour, dtype=np.int32)
    return set(zip(np.minimum(tour, np.roll(tour, -1)),
                   np.maximum(tour, np.roll(tour, -1))))
for seed in SEEDS:
    native = T.run_lkh(str(NATIVE_LKH), str(warm_problem),
                       {"CANDIDATE_SET_TYPE": "POPMUSIC", "MAX_CANDIDATES": 5,
                        "SUBGRADIENT": "NO", "SEED": seed, "MAX_TRIALS": 1,
                        "TIME_LIMIT": 30, "STOP_AT_OPTIMUM": "NO"},
                       str(WORK_ROOT / "warmup" / f"native_{seed}_{uuid.uuid4().hex}"), timeout=30)
    patch_first = np.asarray(read_tour(warm_dir / f"seed_{seed}" / "first.tour")) - 1
    assert edge_set(native["tour"]) == edge_set(patch_first), f"İlk tur uyuşmuyor: {seed}"
print("GPU, yama, altı süreç ve ilk tur eşitliği doğrulandı.")


GPU, yama, altı süreç ve ilk tur eşitliği doğrulandı.


In [ ]:
INSTANCES = []
for group, n, count, distribution, generator_seed, budget_s in INSTANCE_SPECS:
    rng = np.random.default_rng(generator_seed)
    for index in range(count):
        xy = T.to_int_coords(T.gen_instance(n, distribution, rng))
        name = f"augment_{group}_{index:03d}"
        digest = hashlib.sha256(np.ascontiguousarray(xy).tobytes()).hexdigest()
        assert digest == EXPECTED_XY_SHA256[name], f"Eski v8 koordinatları uyuşmuyor: {name}"
        INSTANCES.append({"name": name, "group": group, "n": n,
                          "budget_s": budget_s, "xy": xy, "xy_sha256": digest})
INSTANCES.sort(key=lambda item: (item["budget_s"], item["name"]))
assert len(INSTANCES) == 8
def arm_order(name):
    return sorted(ARMS, key=lambda arm: hashlib.sha256((name + ":" + arm).encode()).hexdigest())
print("Pilot:", len(INSTANCES), "örnek,", len(ARMS), "kol; kesintisiz tek kol bütçesi",
      sum(item["budget_s"] for item in INSTANCES) / 3600, "saat")


Pilot: 8 örnek, 6 kol; kesintisiz tek kol bütçesi 0.4666666666666667 saat


In [ ]:
def arm_record_path(inst, arm):
    return OUTPUT_ROOT / "instances" / inst["name"] / arm / "result.json"

def validate_saved(inst, arm, record):
    assert record["complete"] and record["arm"] == arm and record["instance"] == inst["name"]
    assert record["xy_sha256"] == inst["xy_sha256"] and record["seeds"] == SEEDS
    assert record["budget_s"] == inst["budget_s"] and record["build"] == BUILD_INFO
    assert len(record["seed_costs"]) == 6 and min(record["seed_costs"]) == record["best_cost"]
    return record

def save_progress(state="running", last=None):
    completed = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
    write_json_atomic(OUTPUT_ROOT / "progress.json", {"state": state, "completed_arms": completed,
                      "total_arms": len(INSTANCES) * len(ARMS), "last": last,
                      "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})

new_arms = 0
save_progress()
for inst in INSTANCES:
    instance_dir = OUTPUT_ROOT / "instances" / inst["name"]
    instance_dir.mkdir(parents=True, exist_ok=True)
    coord_path = instance_dir / "xy.npz"
    if not coord_path.exists():
        temp = instance_dir / "xy.tmp.npz"
        np.savez_compressed(temp, xy=inst["xy"])
        os.replace(temp, coord_path)
    with np.load(coord_path, allow_pickle=False) as saved_xy:
        assert hashlib.sha256(np.ascontiguousarray(saved_xy["xy"]).tobytes()).hexdigest() == inst["xy_sha256"]
    local_problem = WORK_ROOT / "problems" / (inst["name"] + ".tsp")
    local_problem.parent.mkdir(exist_ok=True)
    T.write_tsp(local_problem, inst["xy"])
    for arm in arm_order(inst["name"]):
        record_path = arm_record_path(inst, arm)
        if record_path.exists():
            validate_saved(inst, arm, json.loads(record_path.read_text()))
            print("ATLA:", inst["name"], arm, flush=True)
            continue
        if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
            break
        attempt_id = uuid.uuid4().hex
        local_run = WORK_ROOT / "runs" / inst["name"] / arm / attempt_id
        local_run.parent.mkdir(parents=True, exist_ok=True)
        try:
            print("BAŞLA:", inst["name"], arm, "bütçe", inst["budget_s"], "sn", flush=True)
            if arm.startswith("native_"):
                raw = run_native_arm(arm, NATIVE_LKH, local_problem, inst["xy"],
                                     local_run, SEEDS, inst["budget_s"], T)
            else:
                raw = run_live_arm(arm, PATCHED_LKH, local_problem, inst["xy"],
                                   local_run, SEEDS, inst["budget_s"], MODEL, DEVICE,
                                   T, M, sparse_fraction=SPARSE_FRACTION)
            drive_arm = record_path.parent
            drive_arm.mkdir(parents=True, exist_ok=True)
            audit_rel = f"instances/{inst['name']}/{arm}/{attempt_id}"
            audit_dest = OUTPUT_ROOT / audit_rel
            shutil.copytree(local_run, audit_dest)
            local_policy = local_run.parent / (local_run.name + "_audit")
            if local_policy.exists():
                shutil.copytree(local_policy, audit_dest / "policy_audit")
            record = {"schema": 1, "complete": True, "instance": inst["name"],
                      "group": inst["group"], "n": inst["n"], "xy_sha256": inst["xy_sha256"],
                      "arm": arm, "seeds": SEEDS, "budget_s": inst["budget_s"],
                      "seed_costs": raw["seed_costs"], "best_cost": raw["best_cost"],
                      "wall_elapsed_s": raw["wall_elapsed_s"], "build": BUILD_INFO,
                      "hardware": HARDWARE,
                      "audit_relpath": audit_rel, "attempt_id": attempt_id,
                      "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
            validate_saved(inst, arm, record)
            write_json_atomic(record_path, record)
            new_arms += 1
            save_progress(last=inst["name"] + "/" + arm)
            print("BİTTİ:", inst["name"], arm, "en iyi", raw["best_cost"], flush=True)
        except Exception as exc:
            write_json_atomic(OUTPUT_ROOT / "errors" / (inst["name"] + "_" + arm + "_" + attempt_id + ".json"),
                              {"instance": inst["name"], "arm": arm, "error": repr(exc),
                               "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
            save_progress(state="error", last=inst["name"] + "/" + arm)
            raise
    if MAX_NEW_ARMS is not None and new_arms >= MAX_NEW_ARMS:
        break
completed_count = sum(arm_record_path(inst, arm).exists() for inst in INSTANCES for arm in ARMS)
save_progress(state="complete" if completed_count == 48 else "partial")
print(f"Tamamlanan kol: {completed_count}/48. Bu oturumdaki yeni kol: {new_arms}")


BAŞLA: augment_clu10k_000 native_pop5 bütçe 120.0 sn
BİTTİ: augment_clu10k_000 native_pop5 en iyi 33760016
BAŞLA: augment_clu10k_000 native_pop6 bütçe 120.0 sn
BİTTİ: augment_clu10k_000 native_pop6 en iyi 33760199
BAŞLA: augment_clu10k_000 beta_sparse bütçe 120.0 sn
BİTTİ: augment_clu10k_000 beta_sparse en iyi 33759680
BAŞLA: augment_clu10k_000 beta_promote bütçe 120.0 sn
BİTTİ: augment_clu10k_000 beta_promote en iyi 33759992
BAŞLA: augment_clu10k_000 sham bütçe 120.0 sn
BİTTİ: augment_clu10k_000 sham en iyi 33760016
BAŞLA: augment_clu10k_000 frequency_same_gate bütçe 120.0 sn
BİTTİ: augment_clu10k_000 frequency_same_gate en iyi 33759970
BAŞLA: augment_clu10k_001 native_pop5 bütçe 120.0 sn
BİTTİ: augment_clu10k_001 native_pop5 en iyi 16202132
BAŞLA: augment_clu10k_001 beta_promote bütçe 120.0 sn
BİTTİ: augment_clu10k_001 beta_promote en iyi 16201854
BAŞLA: augment_clu10k_001 native_pop6 bütçe 120.0 sn
BİTTİ: augment_clu10k_001 native_pop6 en iyi 16201186
BAŞLA: augment_clu10k_001 sham 

In [ ]:
rows = []
for inst in INSTANCES:
    records = {}
    for arm in ARMS:
        path = arm_record_path(inst, arm)
        if path.exists():
            records[arm] = validate_saved(inst, arm, json.loads(path.read_text()))
    if len(records) == len(ARMS):
        row = {"instance": inst["name"], "group": inst["group"],
               "n": inst["n"], "budget_s": inst["budget_s"],
               "mixed_hardware": len({json.dumps(records[arm]["hardware"], sort_keys=True) for arm in ARMS}) > 1}
        row.update({arm: records[arm]["best_cost"] for arm in ARMS})
        rows.append(row)
table = pd.DataFrame(rows)
summary_dir = OUTPUT_ROOT / "summary"
summary_dir.mkdir(exist_ok=True)
table.to_csv(summary_dir / "complete_instances.csv", index=False)
comparisons = []
if not table.empty and table["mixed_hardware"].any():
    print("UYARI: Bazı örneklerin kolları farklı CPU/kütüphane ortamlarında tamamlandı. Bunları politika seçiminde kullanmayın:",
          table.loc[table["mixed_hardware"], "instance"].tolist())
analysis_table = table.loc[~table["mixed_hardware"]] if not table.empty else table
for treatment in ("beta_promote", "beta_sparse", "frequency_same_gate", "sham"):
    for control in ("native_pop5", "native_pop6", "sham"):
        if treatment == control or analysis_table.empty:
            continue
        difference = 100.0 * (analysis_table[treatment].to_numpy() - analysis_table[control].to_numpy()) / analysis_table[control].to_numpy()
        wins, losses = int((difference < 0).sum()), int((difference > 0).sum())
        comparisons.append({"treatment": treatment, "control": control,
                            "n_instances": len(difference), "wins": wins, "losses": losses,
                            "ties": int((difference == 0).sum()),
                            "mean_paired_pct": float(difference.mean()),
                            "median_paired_pct": float(np.median(difference)),
                            "exploratory_sign_p": float(binomtest(wins, wins + losses).pvalue) if wins + losses else 1.0})
comparison_table = pd.DataFrame(comparisons)
comparison_table.to_csv(summary_dir / "exploratory_comparisons.csv", index=False)
write_json_atomic(summary_dir / "pilot_note.json",
                  {"completed_instances": len(rows), "planned_instances": 8,
                   "same_hardware_instances_for_comparison": len(analysis_table),
                   "interpretation": "Exploratory development pilot on reused v8 cases; no final significance claim.",
                   "next_step": "Freeze one policy before a disjoint new holdout."})
display(table)
display(comparison_table)


,instance,group,n,budget_s,mixed_hardware,native_pop5,native_pop6,sham,beta_promote,beta_sparse,frequency_same_gate
0,augment_clu10k_000,clu10k,10000,120.0,False,33760016,33760199,33760016,33759992,33759680,33759970
1,augment_clu10k_001,clu10k,10000,120.0,False,16202132,16201186,16202132,16201854,16200909,16201527
2,augment_uni10k_000,uni10k,10000,120.0,False,72091465,72087781,72091465,72091555,72091321,72100096
3,augment_uni10k_001,uni10k,10000,120.0,False,71661419,71648570,71661419,71660595,71653122,71650625
4,augment_clu20k_000,clu20k,20000,300.0,False,58590406,58586787,58590496,58595996,58591702,58597346
5,augment_clu20k_001,clu20k,20000,300.0,False,51651804,51650231,51651804,51649340,51646418,51645100
6,augment_uni20k_000,uni20k,20000,300.0,False,101464170,101470181,101464170,101474611,101469111,101467299
7,augment_uni20k_001,uni20k,20000,300.0,False,101221251,101207058,101221251,101214976,101220564,101219948


,treatment,control,n_instances,wins,losses,ties,mean_paired_pct,median_paired_pct,exploratory_sign_p
0,beta_promote,native_pop5,8,5,3,0,0.000756,-0.000610,0.726562
1,beta_promote,native_pop6,8,2,6,0,0.006464,0.004801,0.289062
2,beta_promote,sham,8,5,3,0,0.000737,-0.000610,0.726562
3,beta_sparse,native_pop5,8,6,2,0,-0.003043,-0.000837,0.289062
4,beta_sparse,native_pop6,8,4,4,0,0.002664,0.001928,1.000000
5,beta_sparse,sham,8,6,2,0,-0.003062,-0.000837,0.289062
6,frequency_same_gate,native_pop5,8,5,3,0,-0.000787,-0.000712,0.726562
7,frequency_same_gate,native_pop6,8,3,5,0,0.004920,0.002486,0.726562
8,frequency_same_gate,sham,8,5,3,0,-0.000806,-0.000712,0.726562
9,sham,native_pop5,8,0,1,7,0.000019,0.000000,1.000000


In [ ]:
def checked_zip(name, include_audit):
    local = WORK_ROOT / name
    temp = WORK_ROOT / (name + ".tmp")
    files = []
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6, allowZip64=True) as archive:
        for path in sorted(OUTPUT_ROOT.rglob("*")):
            if not path.is_file() or path.suffix == ".zip" or path.name.endswith(".tmp") or path.name.endswith(".tmp.npz"):
                continue
            rel = path.relative_to(OUTPUT_ROOT)
            if not include_audit and ("policy_audit" in rel.parts or path.name in
                                      {"xy.npz", "native.candidates", "live.candidates", "post.candidates", "first.tour", "best.tour", "stdout.txt", "stderr.txt", "policy_arrays.npz"}):
                continue
            archive.write(path, str(rel), compress_type=zipfile.ZIP_STORED if path.suffix == ".npz" else zipfile.ZIP_DEFLATED)
            files.append(str(rel))
    with zipfile.ZipFile(temp) as check:
        assert check.testzip() is None and len(check.namelist()) == len(files)
    os.replace(temp, local)
    target = OUTPUT_ROOT / "archives" / name
    tmp_target = target.with_name(target.name + ".tmp")
    shutil.copy2(local, tmp_target)
    os.replace(tmp_target, target)
    assert sha256_file(local) == sha256_file(target)
    return {"local": str(local), "drive": str(target), "sha256": sha256_file(local),
            "bytes": local.stat().st_size, "files": len(files)}

complete = len(rows) == 8
audit_zip = checked_zip(EXPERIMENT_NAME + "_audit.zip", True) if complete else None
results_zip = checked_zip(EXPERIMENT_NAME + "_results.zip", False)
write_json_atomic(OUTPUT_ROOT / "archives" / "archive_info.json",
                  {"complete": complete, "results": results_zip, "audit": audit_zip})
print("Sonuç ZIP'i:", results_zip)
if complete:
    print("Tam denetim ZIP'i Drive'da:", audit_zip)
    from google.colab import files
    files.download(results_zip["local"])
else:
    print(f"Ara sonuçlar Drive'da; tamamlanan örnek {len(rows)}/8. Yeniden çalıştırınca devam eder.")


Sonuç ZIP'i: {'local': '/content/betatour_v9_live_work/v9_live_pilot_results.zip', 'drive': '/content/drive/MyDrive/BetaTour/v9_live_pilot/archives/v9_live_pilot_results.zip', 'sha256': '9205b03a8860006f60aae06184f3ca1d8575c511ba23881fdb971065f5f90522', 'bytes': 9740597, 'files': 971}
Tam denetim ZIP'i Drive'da: {'local': '/content/betatour_v9_live_work/v9_live_pilot_audit.zip', 'drive': '/content/drive/MyDrive/BetaTour/v9_live_pilot/archives/v9_live_pilot_audit.zip', 'sha256': 'b0c4dfd5c53c9637c1bfb63dad9550ec753c22e27b38f8990b5314e93b8dd3f5', 'bytes': 339848073, 'files': 2675}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>